# Olist E-Commerce Project Starter

Use this starter notebook for the BAN 6003 final project option: **Olist Brazilian E-Commerce**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset is relational. Your main job is to preserve the row meaning of your final ABT.

- A common final ABT is **one row per order**, identified by `order_id`.
- Aggregate `order_items`, `order_payments`, and `order_reviews` to `order_id` before joining them to `orders`.
- Join `products`, category translations, and sellers to item-level data before aggregating those fields to order level.
- `customer_id` identifies the customer record attached to one order; `customer_unique_id` may repeat because one person can place multiple orders.
- Check row counts and duplicate `order_id` values after every major merge.
- Define targets only where the outcome is observable. An undelivered order is not automatically an on-time order, and a missing review is not a low review.
- State when a prediction would be made, then exclude fields that would not be available at that time.


## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
orders = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
customers = pd.read_csv(DATA_DIR / 'customers.csv')
items = pd.read_csv(DATA_DIR / 'order_items.csv', parse_dates=['shipping_limit_date'])
payments = pd.read_csv(DATA_DIR / 'order_payments.csv')
reviews = pd.read_csv(DATA_DIR / 'order_reviews.csv', parse_dates=['review_creation_date', 'review_answer_timestamp'])
products = pd.read_csv(DATA_DIR / 'products.csv')
sellers = pd.read_csv(DATA_DIR / 'sellers.csv')
geo = pd.read_csv(DATA_DIR / 'geolocation_zip_prefix.csv')
translations = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

tables = {
    'orders': orders,
    'customers': customers,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation_zip_prefix': geo,
    'translations': translations,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(5))


orders: 25,000 rows x 8 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28



customers: 25,000 rows x 5 columns


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
2,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
3,fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
4,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR



items: 28,343 rows x 7 columns


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
1,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69
2,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85
3,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65
4,0009792311464db532ff765bf7b182ae,1,8cab8abac59158715e0d70a36c807415,530ec6109d11eaaf87999465c6afee01,2018-08-17 12:15:10,99.90,27.65


... (8 outputs left out)


geolocation_zip_prefix: 10,009 rows x 6 columns


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_records
0,1001,-23.550190,-46.634024,sao paulo,SP,26
1,1004,-23.549799,-46.634757,sao paulo,SP,22
2,1005,-23.549456,-46.636733,sao paulo,SP,25
3,1006,-23.550102,-46.636137,sao paulo,SP,9
4,1008,-23.546002,-46.635886,sao paulo,SP,16



translations: 71 rows x 2 columns


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,orders,25000,8,0,1266
1,customers,25000,5,0,0
2,items,28343,7,0,0
3,payments,26154,5,0,0
4,reviews,24934,7,0,36618
5,products,13344,9,0,924
6,sellers,2234,4,0,0
7,geolocation_zip_prefix,10009,6,0,0
8,translations,71,2,0,0


### Key and Granularity Audit

Confirm the expected key for each table before planning joins. A duplicate `order_id` is expected in item, payment, and review detail tables because those tables can contain several records for one order. Their more detailed keys should still be checked. The provided geolocation table is already summarized to one row per zip-code prefix.


In [5]:
key_audit = pd.DataFrame([
    {"table": "orders", "expected_key": "order_id", "duplicate_keys": orders["order_id"].duplicated().sum(), "missing_key_values": orders["order_id"].isna().sum()},
    {"table": "customers", "expected_key": "customer_id", "duplicate_keys": customers["customer_id"].duplicated().sum(), "missing_key_values": customers["customer_id"].isna().sum()},
    {"table": "order_items", "expected_key": "order_id + order_item_id", "duplicate_keys": items.duplicated(["order_id", "order_item_id"]).sum(), "missing_key_values": items[["order_id", "order_item_id"]].isna().any(axis=1).sum()},
    {"table": "order_payments", "expected_key": "order_id + payment_sequential", "duplicate_keys": payments.duplicated(["order_id", "payment_sequential"]).sum(), "missing_key_values": payments[["order_id", "payment_sequential"]].isna().any(axis=1).sum()},
    {"table": "order_reviews", "expected_key": "review record (review_id may repeat)", "duplicate_keys": reviews.duplicated().sum(), "missing_key_values": reviews["order_id"].isna().sum()},
    {"table": "products", "expected_key": "product_id", "duplicate_keys": products["product_id"].duplicated().sum(), "missing_key_values": products["product_id"].isna().sum()},
    {"table": "sellers", "expected_key": "seller_id", "duplicate_keys": sellers["seller_id"].duplicated().sum(), "missing_key_values": sellers["seller_id"].isna().sum()},
    {"table": "geolocation_zip_prefix", "expected_key": "geolocation_zip_code_prefix", "duplicate_keys": geo["geolocation_zip_code_prefix"].duplicated().sum(), "missing_key_values": geo["geolocation_zip_code_prefix"].isna().sum()},
    {"table": "translations", "expected_key": "product_category_name", "duplicate_keys": translations["product_category_name"].duplicated().sum(), "missing_key_values": translations["product_category_name"].isna().sum()},
])

display(key_audit)
print(f"Item rows sharing an order_id: {items['order_id'].duplicated().sum():,}")
print(f"Payment rows sharing an order_id: {payments['order_id'].duplicated().sum():,}")
print(f"Review rows sharing an order_id: {reviews['order_id'].duplicated().sum():,}")
print(f"Repeated customer_unique_id values: {customers['customer_unique_id'].duplicated().sum():,}")


,table,expected_key,duplicate_keys,missing_key_values
0,orders,order_id,0,0
1,customers,customer_id,0,0
2,order_items,order_id + order_item_id,0,0
3,order_payments,order_id + payment_sequential,0,0
4,order_reviews,review record (review_id may repeat),0,0
5,products,product_id,0,0
6,sellers,seller_id,0,0
7,geolocation_zip_prefix,geolocation_zip_code_prefix,0,0
8,translations,product_category_name,0,0


Item rows sharing an order_id: 3,552
Payment rows sharing an order_id: 1,154
Review rows sharing an order_id: 137
Repeated customer_unique_id values: 236


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Olist notes:

- Keep a table inventory: what does one row mean in each file?
- `order_items`, `order_payments`, and `order_reviews` are not guaranteed to be one row per order.
- Missing review comments are common and should be described, not automatically deleted.
- Product categories may need translation using `product_category_name_translation.csv`.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### 

- Tables/files used: orders, customers, order_items, products, sellers, and order_payments csv files will be used for the analysis. product_category_name_translation.cvs may also be used to translate product categories. orders.csv will serve as the base table because the final analysis will focus on one row per order.

- Early data quality issues: Initial profiling identified several data quality issues that may need further investigation. ZIP code prefix columns in the customers, sellers, and geolocation tables are stored as integers even though they represent geographic codes. Some delivery-related date columns in orders contain missing values. Some item prices, freight values, and payment values may be unusually high or require further inspection. The reviews table also contains 56 repeated review_id values, although these should not automatically be treated as duplicates. Categorical values were checked using value_counts(), and no obvious inconsistencies were identified.

- Initial cleaning plan: ZIP code prefix columns will be converted to string type. Missing delivery-related dates will be compared with order_status before deciding how to handle them. Unusual price, freight, and payment values will be investigated before deciding whether cleaning is necessary. Repeated review_id records will also be inspected with their associated order_id values before any duplicate handling is performed.

- Possible target/outcome: The primary analytical target is late-delivery risk at the order level, while the potential business users are fulfillment, operations, and supply chain teams. I expect longer shipping distances and cross-state deliveries to be associated with a higher risk of late delivery. Product and order characteristics, such as product size, weight, and number of items, may also be related to delivery risk. In addition, some geographic areas may experience higher late-delivery rates than others.

- Ethics, privacy, or governance concerns: This data analysis should be conducted responsibly by avoiding unfair assumptions about customers or sellers. Customer and seller location information should be handled carefully to protect privacy, and all data cleaning and transformation decisions should be clearly documented to maintain proper data governance.

## 1. Business question and Analytical question

- Business or research question: What factors are associated with late delivery in an e-commerce marketplace, and how can these insights help identify higher-risk orders and geographic areas that may benefit from additional fulfillment capacity?

- Why this question matters: Late delivery is an important supply chain and operations issue because it can affect both customer experience and fulfillment efficiency. I became interested in this topic after speaking with H-E-B representatives at a recent career fair, where I learned about the company’s ongoing supply chain expansion, including new fulfillment, warehouse, and distribution facilities. We also discussed how business teams use data when making expansion decisions, such as evaluating where additional capacity may be needed and whether a location can effectively support operations. This made me curious about how I could approach a similar problem as a data analyst.
Through this project, I want to explore which factors are associated with late delivery, which orders may have higher delivery risk, and whether there are geographic patterns in delivery challenges. Although the dataset is not related to H-E-B, it allows me to explore how similar types of data analysis can be used to better understand fulfillment and supply chain operations.



## 2. Understanding the Olist Data

Before starting the data profiling, I first reviewed the Olist dataset documentation to understand what each table represents and how the tables are related.

The dataset contains multiple tables covering different parts of the e-commerce process:

- `orders`: one row represents one order and contains order status and purchase/delivery timestamps.
- `customers`: one row represents an order-specific customer record and contains customer location information.
- `order_items`: one row represents an item within an order and contains product, seller, price, and freight information.
- `order_payments`: one row represents a payment record associated with an order.
- `order_reviews`: one row represents a review record associated with an order.
- `products`: one row represents a product and contains product category, weight, and dimension information.
- `sellers`: one row represents a seller and contains seller location information.
- `geolocation`: contains geographic information based on ZIP-code prefixes.
- `category_translation`: provides English translations for product category names.

These tables have different levels of detail. For example, the `orders` table contains one row per order, while `order_items` and `order_payments` can contain multiple rows for the same order.

Since my project focuses on late delivery, the `orders` table will be the main starting point for the analysis. The other tables may later provide additional information about customers, products, sellers, and geographic locations that could help explain delivery patterns.

Before deciding how these tables should be cleaned or combined, I first need to examine their structure and data quality.

## 3. Initial Data Profiling

In [6]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


orders


,dtype
order_id,str
customer_id,str
order_status,str
order_purchase_timestamp,datetime64[us]
order_approved_at,datetime64[us]
order_delivered_carrier_date,datetime64[us]
order_delivered_customer_date,datetime64[us]
order_estimated_delivery_date,datetime64[us]


,missing_rate
order_delivered_customer_date,0.03100
order_delivered_carrier_date,0.01808
order_approved_at,0.00156
order_id,0.00000
customer_id,0.00000
order_status,0.00000
order_purchase_timestamp,0.00000
order_estimated_delivery_date,0.00000



customers


,dtype
customer_id,str
customer_unique_id,str
customer_zip_code_prefix,int64
customer_city,str
customer_state,str


,missing_rate
customer_id,0.0
customer_unique_id,0.0
customer_zip_code_prefix,0.0
customer_city,0.0
customer_state,0.0


... (17 outputs left out)

,missing_rate
geolocation_zip_code_prefix,0.0
geolocation_lat,0.0
geolocation_lng,0.0
geolocation_city,0.0
geolocation_state,0.0
geolocation_records,0.0



translations


,dtype
product_category_name,str
product_category_name_english,str


,missing_rate
product_category_name,0.0
product_category_name_english,0.0


### 1) Profiling table [orders] 

### Note
- One row in the Orders table represents one order. 
- The table has 25,000 rows and 8 columns. 
- The data types appear appropriate, but there are some missing values in the date columns. 
- order_id is unique with no missing values, and there are no fully duplicated rows. 
- order_status has 8 unique categories. 
- The min and max values of order_purchase_timestamp show a reasonable date range, with no obvious suspicious values identified so far. 
- Further investigation is needed for the missing values and other possible data quality issues.

In [7]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28


In [8]:
orders.tail()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
24995,21a00b08cbeb5716bbb66105e3dbd850,002b5342c72978cf0aba6aae1f5d5293,canceled,2018-09-10 17:51:25,NaT,NaT,NaT,2018-09-20
24996,87ae60ef8b08ae0e5f903cacb53a6904,fea4d57ed3a45455f89c25ef3dae8ee8,canceled,2018-09-11 16:45:54,NaT,NaT,NaT,2018-09-26
24997,ea844c92cf978ea23321fa7fe5871761,6ddc45ad08e6a2c885ca2255fc52d6df,canceled,2018-09-13 09:56:12,NaT,NaT,NaT,2018-09-27
24998,392ed9afd714e3c74767d0c4d3e3f477,2823ffda607a2316375088e0d00005ec,canceled,2018-09-29 09:13:03,NaT,NaT,NaT,2018-10-15
24999,616fa7d4871b87832197b2a137a115d2,bf6181a85bbb4115736c0a8db1a53be3,canceled,2018-10-01 15:30:09,NaT,NaT,NaT,2018-10-23


In [9]:
orders.shape

(25000, 8)

In [10]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       25000 non-null  str           
 1   customer_id                    25000 non-null  str           
 2   order_status                   25000 non-null  str           
 3   order_purchase_timestamp       25000 non-null  datetime64[us]
 4   order_approved_at              24961 non-null  datetime64[us]
 5   order_delivered_carrier_date   24548 non-null  datetime64[us]
 6   order_delivered_customer_date  24225 non-null  datetime64[us]
 7   order_estimated_delivery_date  25000 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 3.3 MB


In [11]:
orders.isna().sum()

order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 39
order_delivered_carrier_date     452
order_delivered_customer_date    775
order_estimated_delivery_date      0
dtype: int64

In [12]:
orders["order_status"].nunique()

8

In [13]:
orders["order_status"].value_counts()

order_status
delivered      24222
shipped          309
unavailable      167
canceled         154
invoiced          75
processing        71
approved           1
created            1
Name: count, dtype: int64

In [14]:
orders.duplicated().sum()

np.int64(0)

In [15]:
orders["order_id"].duplicated().sum()

np.int64(0)

In [16]:
orders["order_purchase_timestamp"].min()

Timestamp('2016-10-03 09:44:50')

In [17]:
orders["order_purchase_timestamp"].max()

Timestamp('2018-10-01 15:30:09')

### 2) Profiling table [customers]
### Note
- One row in the Customers table represents a customer record for one order. If the same customer places another order, a new customer_id is created.
- The table has 25,000 rows and 5 columns. 
- There are no missing values or fully duplicated rows, and customer_id is unique with no duplicates. However, customer_unique_id has repeated values because the same customer can place multiple orders. 
- customer_zip_code_prefix is currently stored as an integer, but it represents a geographic code rather than a numeric measurement, so converting it to a string should be considered during cleaning.

In [18]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
2,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
3,fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
4,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR


In [19]:
customers.tail()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
24995,c156d63bdfce1d456bd43cf1c4dadfca,e5cf9da72b751849b7b217f7a943d0fa,9060,santo andre,SP
24996,a1b5ca506b592bb72d4caadcbfe71385,c811311629fd3d76f5ca5eeb2abfa7e8,17511,marilia,SP
24997,935993f47af1ed7d0715c26b686341c5,4452b8ef472646c4cc042cb31a291f3b,12236,sao jose dos campos,SP
24998,592b8900e0e8325027d885e6d30d0283,57c2cfb4a80b13ed19b5fb258d29c19d,15720,palmeira d'oeste,SP
24999,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS


In [20]:
customers.shape

(25000, 5)

In [21]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               25000 non-null  str  
 1   customer_unique_id        25000 non-null  str  
 2   customer_zip_code_prefix  25000 non-null  int64
 3   customer_city             25000 non-null  str  
 4   customer_state            25000 non-null  str  
dtypes: int64(1), str(4)
memory usage: 2.8 MB


In [22]:
customers.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [23]:
customers["customer_id"].isna().sum()

np.int64(0)

In [24]:
customers["customer_state"].nunique()

27

In [25]:
customers.duplicated().sum()

np.int64(0)

In [26]:
customers["customer_id"].duplicated().sum()

np.int64(0)

### 3) Profiling table [items]
### note
- One row represents one item line within an order. 
- The Items table has 28,343 rows and 7 columns. 
- It has no missing values or fully duplicated rows. 
- The range of shipping_limit_date is from 2016 to 2020, which seems reasonable. However, the ranges of price (2.20 to 6,735.00) and freight_value (0 to 339.59) may need further inspection for possible unusual values.

In [27]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
1,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69
2,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85
3,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65
4,0009792311464db532ff765bf7b182ae,1,8cab8abac59158715e0d70a36c807415,530ec6109d11eaaf87999465c6afee01,2018-08-17 12:15:10,99.90,27.65


In [28]:
items.tail()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
28338,fff7c4452f050315db1b3f24d9df5fcd,1,dd469c03ad67e201bc2179ef077dcd48,7e93a43ef30c4f03f38b393420bc753a,2017-06-07 17:05:23,736.00,20.91
28339,fff8287bbae429a99bb7e8c21d151c41,1,bee2e070c39f3dd2f6883a17a5f0da45,4e922959ae960d389249c378d1c939f5,2018-03-27 12:29:22,180.00,48.14
28340,fff8287bbae429a99bb7e8c21d151c41,2,bee2e070c39f3dd2f6883a17a5f0da45,4e922959ae960d389249c378d1c939f5,2018-03-27 12:29:22,180.00,48.14
28341,fffbee3b5462987e66fb49b1c5411df2,1,6f0169f259bb0ff432bfff7d829b9946,213b25e6f54661939f11710a6fddb871,2018-06-28 09:58:03,119.85,20.03
28342,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53


In [29]:
items.shape

(28343, 7)

In [30]:
items.info()

<class 'pandas.DataFrame'>
RangeIndex: 28343 entries, 0 to 28342
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             28343 non-null  str           
 1   order_item_id        28343 non-null  int64         
 2   product_id           28343 non-null  str           
 3   seller_id            28343 non-null  str           
 4   shipping_limit_date  28343 non-null  datetime64[us]
 5   price                28343 non-null  float64       
 6   freight_value        28343 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 4.1 MB


In [31]:
items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [32]:
items.duplicated().sum()

np.int64(0)

In [33]:
items["shipping_limit_date"].min()

Timestamp('2016-10-08 10:34:01')

In [34]:
items["shipping_limit_date"].max()

Timestamp('2020-04-09 22:35:08')

In [35]:
items["price"].min()

np.float64(2.2)

In [36]:
items["price"].max()

np.float64(6735.0)

In [37]:
items["freight_value"].min()

np.float64(0.0)

In [38]:
items["freight_value"].max()

np.float64(339.59)

### 4) Profiling table [payments]
### note
- One row represents one payment transaction or payment method sequence within an order. 
- The Payments table has 26,154 rows and 5 columns. 
- There are no obvious data type or missing value issues. payment_type has 4 categories. 
- The payment_value ranges from 0 to 6,929.31, so the minimum and maximum values may need further inspection for possible unusual values.

In [39]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
1,0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95
2,cf95215a722f3ebf29e6bbab87a29e61,1,credit_card,5,102.66
3,61059985a6fc0ad64e95d9944caacdad,1,credit_card,1,132.04
4,b2349a3f20dfbeef62e7b31baa22f84b,1,credit_card,3,136.71


In [40]:
payments.tail()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
26149,0872d0faafe0cb56e6f2c594cca1522c,1,credit_card,2,127.92
26150,2e3d96522c8c378d4135c12a65c977aa,1,credit_card,4,47.62
26151,7159096c5aa9be77f7f0c26c01ee9793,1,credit_card,4,280.65
26152,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54
26153,28bbae6599b09d39ca406b747b6632b1,1,boleto,1,191.58


In [41]:
payments.shape

(26154, 5)

In [42]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 26154 entries, 0 to 26153
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   order_id              26154 non-null  str    
 1   payment_sequential    26154 non-null  int64  
 2   payment_type          26154 non-null  str    
 3   payment_installments  26154 non-null  int64  
 4   payment_value         26154 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 2.0 MB


In [43]:
payments.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [44]:
payments.duplicated().sum()

np.int64(0)

In [45]:
payments["payment_type"].nunique()

4

In [46]:
payments["payment_type"].value_counts()

payment_type
credit_card    19302
boleto          4973
voucher         1488
debit_card       391
Name: count, dtype: int64

In [47]:
payments["payment_installments"].value_counts()

payment_installments
1     13147
2      3093
3      2665
4      1843
10     1372
5      1300
8      1095
6       955
7       437
9       170
12       31
15       14
24        7
18        6
13        5
20        5
21        2
11        2
14        2
16        2
17        1
Name: count, dtype: int64

In [48]:
payments["payment_value"].min()

np.float64(0.0)

In [49]:
payments["payment_value"].max()

np.float64(6929.31)

### 5) Profiling table [reviews]
### note
- One row represents one review record associated with an order. 
- The Reviews table has 24,934 rows and 7 columns. 
- The data types and missing values seem reasonable, and there are no fully duplicated rows. 
- review_id has 56 duplicated values, but review_id is not guaranteed to be unique in this table, so further inspection is needed before treating them as duplicate issues. 
- The ranges of review_creation_date and review_answer_timestamp also seem reasonable.

In [50]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24
1,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53
2,7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,NaN,NaN,2018-08-14,2018-08-14 21:36:06
3,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...,2018-05-22,2018-05-23 16:45:47
4,8e15a274d95600fa14f8be64e37a0e67,ff1581e08b3011021e7c7de592ddc81e,5,NaN,NaN,2018-03-24,2018-03-26 15:58:32


In [51]:
reviews.tail()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
24929,6c5fa14b1cb0ede918c75b1fabb222a5,10a3f7e9bf88b7dc86a767fd278e5bd2,5,NaN,NaN,2017-11-21,2017-11-23 08:28:00
24930,586e77c5f02b95285a939083359f8241,5f4a120717a4d152577d7d2ec66c45be,5,NaN,NaN,2018-08-05,2018-08-06 09:48:41
24931,c2642a756d89f7e781ca0180829770bb,652050879a7da861024a1114f926958d,5,NaN,NaN,2018-06-08,2018-06-08 22:22:23
24932,3ea12859f456855149b564d1a221f87a,f62f01ec64c6b221f926ba560907d8f7,3,NaN,NaN,2017-04-05,2017-04-05 09:42:58
24933,af2dc0519de6e0720ef0c74292fb4114,d699c734a0b1c8111f2272a3f36d398c,5,NaN,NaN,2018-04-27,2018-04-30 01:18:57


In [52]:
reviews.shape

(24934, 7)

In [53]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 24934 entries, 0 to 24933
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                24934 non-null  str           
 1   order_id                 24934 non-null  str           
 2   review_score             24934 non-null  int64         
 3   review_comment_title     2926 non-null   str           
 4   review_comment_message   10324 non-null  str           
 5   review_creation_date     24934 non-null  datetime64[us]
 6   review_answer_timestamp  24934 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(4)
memory usage: 3.6 MB


In [54]:
reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       22008
review_comment_message     14610
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [55]:
reviews.duplicated().sum()

np.int64(0)

In [56]:
reviews["review_id"].duplicated().sum()

np.int64(56)

In [57]:
reviews["review_creation_date"].min()

Timestamp('2016-10-15 00:00:00')

In [58]:
reviews["review_creation_date"].max()

Timestamp('2018-08-31 00:00:00')

In [59]:
reviews["review_answer_timestamp"].min()

Timestamp('2016-10-16 03:20:17')

In [60]:
reviews["review_answer_timestamp"].max()

Timestamp('2018-10-29 12:27:35')

### 6) Profiling table [products]
### note
- One row represents one product. 
- The Products table has 13,344 rows and 9 columns. 
- The data types and missing values seem reasonable, although product_photos_qty may be converted to int64 after handling its missing values. 
- product_category_name has 72 categories. 
- The ranges of the numerical columns generally seem reasonable, but a few maximum values for product dimensions and weight may need further investigation for possible unusual values.

In [61]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60.0,745.0,1.0,200.0,38.0,5.0,11.0
1,14aa47b7fe5c25522b47b4b29c98dcb9,cama_mesa_banho,54.0,630.0,1.0,1100.0,16.0,10.0,16.0
2,cf55509ea8edaaac1d28fdb16e48fc22,instrumentos_musicais,43.0,1827.0,3.0,250.0,17.0,7.0,17.0
3,6a2fb4dd53d2cdb88e0432f1284a004c,perfumaria,39.0,346.0,2.0,400.0,27.0,5.0,20.0
4,a0736b92e52f6cead290e30b578413b6,moveis_decoracao,56.0,296.0,2.0,1700.0,100.0,7.0,15.0


In [62]:
products.tail()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
13339,dfec64aac9b864b2807a7be33222b75f,cama_mesa_banho,58.0,252.0,4.0,850.0,38.0,7.0,28.0
13340,3becff10d1deb92b02f2a1ee62a04524,informatica_acessorios,54.0,1520.0,2.0,6150.0,30.0,30.0,20.0
13341,c4e71b64511b959455e2107fe7859020,utilidades_domesticas,59.0,1371.0,2.0,200.0,18.0,15.0,15.0
13342,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
13343,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [63]:
products.shape

(13344, 9)

In [64]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 13344 entries, 0 to 13343
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  13344 non-null  str    
 1   product_category_name       13114 non-null  str    
 2   product_name_lenght         13114 non-null  float64
 3   product_description_lenght  13114 non-null  float64
 4   product_photos_qty          13114 non-null  float64
 5   product_weight_g            13343 non-null  float64
 6   product_length_cm           13343 non-null  float64
 7   product_height_cm           13343 non-null  float64
 8   product_width_cm            13343 non-null  float64
dtypes: float64(7), str(2)
memory usage: 1.5 MB


In [65]:
products.isna().sum()

product_id                      0
product_category_name         230
product_name_lenght           230
product_description_lenght    230
product_photos_qty            230
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64

In [66]:
products.duplicated().sum()

np.int64(0)

In [67]:
products["product_id"].duplicated().sum()

np.int64(0)

In [68]:
products["product_category_name"].nunique()

72

In [69]:
products[["product_name_lenght",
"product_description_lenght",
"product_photos_qty", 
"product_weight_g",            
"product_length_cm",           
"product_height_cm",           
"product_width_cm"]].agg(["min","max"])

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
min,5.0,4.0,1.0,2.0,7.0,2.0,7.0
max,76.0,3985.0,18.0,30000.0,105.0,105.0,118.0


### 7) Profiling table [sellers]
### note
- One row represents one seller. 
- The Sellers table has 2,234 rows and 4 columns. 
- There are no missing values. seller_zip_code_prefix is stored as an integer, but it represents a geographic code, so it should be converted to a string during cleaning. 
- seller_state has 21 unique categories.

In [70]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
3,e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
4,1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP


In [71]:
sellers.tail()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
2229,725af026f869236a8dd0ceb98564d156,35530,claudio,MG
2230,7bac63f6603d382cc8d0832eb6c100a8,5713,sao paulo,SP
2231,f1fdf2d13186575751aa25876536d85c,5314,sao paulo,SP
2232,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
2233,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS


In [72]:
sellers.shape

(2234, 4)

In [73]:
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 2234 entries, 0 to 2233
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               2234 non-null   str  
 1   seller_zip_code_prefix  2234 non-null   int64
 2   seller_city             2234 non-null   str  
 3   seller_state            2234 non-null   str  
dtypes: int64(1), str(3)
memory usage: 166.1 KB


In [74]:
sellers.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [75]:
sellers.duplicated().sum()

np.int64(0)

In [76]:
sellers["seller_id"].duplicated().sum()

np.int64(0)

In [77]:
sellers["seller_state"].nunique()

21

In [78]:
sellers["seller_state"].value_counts()

seller_state
SP    1362
PR     247
MG     170
SC     136
RJ     119
RS      84
DF      27
GO      26
ES      16
BA      13
CE       7
PE       6
RN       4
PB       4
MS       4
MT       3
RO       2
PI       1
SE       1
MA       1
PA       1
Name: count, dtype: int64

### 8) Profiling table [geo(geolocation_zip_prefix)]
### note
- One row represents one zip-code-prefix geographic summary. 
- The Geolocation table has 10,009 rows and 6 columns. 
- geolocation_zip_code_prefix is stored as an integer, but it represents a geographic code, so it should be converted to a string during cleaning. The other data types and missing values show no obvious issues.
- geolocation_state has 27 categories. 
- The ranges of latitude (geolocation_lat) and longitude (geolocation_lng) were also checked and may need further inspection to confirm that the geographic coordinates are reasonable.

In [79]:
geo.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_records
0,1001,-23.550190,-46.634024,sao paulo,SP,26
1,1004,-23.549799,-46.634757,sao paulo,SP,22
2,1005,-23.549456,-46.636733,sao paulo,SP,25
3,1006,-23.550102,-46.636137,sao paulo,SP,9
4,1008,-23.546002,-46.635886,sao paulo,SP,16


In [80]:
geo.tail()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_records
10004,99900,-27.886630,-52.228478,getulio vargas,RS,102
10005,99930,-27.914102,-52.258782,estacao,RS,32
10006,99950,-28.068959,-52.012068,tapejara,RS,153
10007,99960,-27.953722,-52.025511,charrua,RS,5
10008,99970,-28.343766,-51.874689,ciriaco,RS,21


In [81]:
geo.shape

(10009, 6)

In [82]:
geo.info()

<class 'pandas.DataFrame'>
RangeIndex: 10009 entries, 0 to 10008
Data columns (total 6 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   geolocation_zip_code_prefix  10009 non-null  int64  
 1   geolocation_lat              10009 non-null  float64
 2   geolocation_lng              10009 non-null  float64
 3   geolocation_city             10009 non-null  str    
 4   geolocation_state            10009 non-null  str    
 5   geolocation_records          10009 non-null  int64  
dtypes: float64(2), int64(2), str(2)
memory usage: 588.4 KB


In [83]:
geo.isna().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
geolocation_records            0
dtype: int64

In [84]:
geo.duplicated().sum()

np.int64(0)

In [85]:
geo["geolocation_state"].nunique()

27

In [86]:
geo["geolocation_state"].value_counts()

geolocation_state
SP    4143
MG     928
RJ     916
PR     594
RS     577
BA     412
SC     353
GO     296
DF     233
PE     232
CE     207
ES     198
MS     109
PA     108
MT     106
MA      93
PB      89
RN      82
PI      71
AL      66
SE      51
TO      48
RO      32
AM      31
AC      12
AP      11
RR      11
Name: count, dtype: int64

In [87]:
geo[["geolocation_lat", "geolocation_lng"]].agg(["min","max"])

,geolocation_lat,geolocation_lng
min,-33.689948,-68.749175
max,7.271649,-34.799347


### 9) Profiling table [translations]
### note
- One row represents one Portuguese to English product category translation. 
- The Translations table has 71 rows and 2 columns. 
- The data types and missing values show no obvious issues, and there are no fully duplicated rows.

In [88]:
translations.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [89]:
translations.shape

(71, 2)

In [90]:
translations.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 3.5 KB


In [91]:
translations.isna().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

In [92]:
translations.duplicated().sum()

np.int64(0)

In [93]:
translations["product_category_name"].duplicated().sum()

np.int64(0)

### 10) Initial Data Profiling Summary

The Olist dataset contains nine tables with different row meanings and levels of granularity. Initial profiling was conducted to examine table dimensions, data types, missing values, duplicates, categorical values, and numerical ranges.

### Orders
- 25,000 rows and 8 columns.
- One row represents one order.
- Data types generally appear appropriate.
- Some date columns contain missing values and require further investigation.
- `order_id` is unique with no missing values.
- No fully duplicated rows were identified.
- `order_status` contains 8 categories.
- The purchase date range appears reasonable.

### Customers
- 25,000 rows and 5 columns.
- Each row represents a customer record for one order.
- No missing values or fully duplicated rows were identified.
- `customer_id` is unique, while `customer_unique_id` can repeat across orders.
- `customer_zip_code_prefix` is stored as an integer but represents a geographic code and may need to be converted to a string.

### Order Items
- 28,343 rows and 7 columns.
- One row represents one item line within an order.
- No missing values or fully duplicated rows were identified.
- The shipping date range appears reasonable.
- Some maximum values in `price` and `freight_value` may require further inspection.

### Payments
- 26,154 rows and 5 columns.
- One row represents one payment transaction or payment method sequence within an order.
- No obvious data type or missing value issues were identified.
- `payment_type` contains 4 categories.
- `payment_value` ranges from 0 to 6,929.31, so extreme values may require further inspection.

### Reviews
- 24,934 rows and 7 columns.
- One row represents one review record associated with an order.
- Data types and missing values generally appear reasonable.
- No fully duplicated rows were identified.
- `review_id` has 56 duplicated values. Further investigation is needed because `review_id` is not guaranteed to be unique.
- Review creation and answer timestamp ranges appear reasonable.

### Products
- 13,344 rows and 9 columns.
- One row represents one product.
- Data types and missing values generally appear reasonable.
- `product_photos_qty` may be converted to an integer after handling its missing values.
- `product_category_name` contains 72 categories.
- Some maximum values for product dimensions and weight may require further inspection.

### Sellers
- 2,234 rows and 4 columns.
- One row represents one seller.
- No missing values were identified.
- `seller_zip_code_prefix` is stored as an integer but represents a geographic code and may need to be converted to a string.
- `seller_state` contains 21 categories.

### Geolocation
- 10,009 rows and 6 columns.
- One row represents one zip-code-prefix geographic summary.
- No obvious missing value issues were identified.
- `geolocation_zip_code_prefix` should be converted from integer to string.
- `geolocation_state` contains 27 categories.
- Latitude and longitude ranges may require further inspection.

### Product Category Translation
- 71 rows and 2 columns.
- One row represents one Portuguese-to-English product category translation.
- No obvious data type or missing value issues were identified.
- No fully duplicated rows were identified.

### 11) Specific Data Quality Issues

### Additional Profiling Check
Categorical values were checked using `value_counts()`, and no obvious inconsistencies were identified.

### Issue 1: ZIP code columns are stored as integers
- **Table / Column:** Customers - `customer_zip_code_prefix`, Sellers - `seller_zip_code_prefix`, Geolocation - `geolocation_zip_code_prefix`
- **Finding:** ZIP code prefixes are stored as `int64`.
- **Concern:** ZIP codes are geographic identifiers rather than numerical measurements.
- **Initial plan:** Convert these columns to string type and verify the values after conversion.

### Issue 2: Missing delivery-related dates
- **Table / Column:** Orders - `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`
- **Finding:** These date columns contain missing values.
- **Concern:** The missing values may be related to order status rather than simple data errors.
- **Initial plan:** Compare the missing values with `order_status` before deciding how they should be handled.

### Issue 3: Potential unusual values in item price and freight value
- **Table / Column:** Order Items - `price`, `freight_value`
- **Finding:** `price` ranges from 2.20 to about 6,735, and `freight_value` ranges from 0 to 339.59.
- **Concern:** Some extreme values may represent unusual observations.
- **Initial plan:** Inspect the distributions and records with extreme values before deciding whether cleaning is necessary.

### Issue 4: Potential unusual payment values
- **Table / Column:** Payments - `payment_value`
- **Finding:** `payment_value` ranges from 0 to 6,929.31.
- **Concern:** Zero payments and very high payment values may require further investigation.
- **Initial plan:** Inspect records with zero and unusually high payment values and compare them with related order information.

### Issue 5: Repeated review IDs
- **Table / Column:** Reviews - `review_id`
- **Finding:** `review_id` contains 56 duplicated values.
- **Concern:** `review_id` is not guaranteed to be unique by itself, so these records should not automatically be treated as duplicates.
- **Initial plan:** Inspect the repeated `review_id` records and their associated `order_id` values before deciding whether any duplicate handling is needed.

### 12) Initial Cleaning Plan

- Convert ZIP code prefix columns in Customers, Sellers, and Geolocation from integer to string type.
- Investigate missing delivery-related dates in Orders by comparing them with `order_status` before deciding how to handle them.
- Inspect unusual values in `price`, `freight_value`, and `payment_value` before deciding whether cleaning is necessary.
- Investigate repeated `review_id` values with their associated `order_id` values before performing any duplicate handling.
- Keep the raw data unchanged and document any cleaning decisions before creating cleaned versions of the data.

## 4. Analytical Design and Data Preparation Plan

### A. Late delivery outcome + usuable population definition
- actual delivery date > estimated delivery date => late delivery

### Note

Late delivery can only be evaluated for orders that were actually delivered and have enough information to compare the actual delivery date with the estimated delivery date. I first reviewed order_status to understand the different order outcomes in the dataset. I then examined missing values in order_delivered_customer_date by order status to determine whether missing actual delivery dates were related to orders that had not been delivered. I also checked order_estimated_delivery_date for missing values because this date is required as the reference point for determining whether an order arrived late. Based on these checks, the late-delivery analysis will focus on delivered orders with both order_delivered_customer_date and order_estimated_delivery_date available.

In [94]:
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28
...,...,...,...,...,...,...,...,...
24995,21a00b08cbeb5716bbb66105e3dbd850,002b5342c72978cf0aba6aae1f5d5293,canceled,2018-09-10 17:51:25,NaT,NaT,NaT,2018-09-20
24996,87ae60ef8b08ae0e5f903cacb53a6904,fea4d57ed3a45455f89c25ef3dae8ee8,canceled,2018-09-11 16:45:54,NaT,NaT,NaT,2018-09-26
24997,ea844c92cf978ea23321fa7fe5871761,6ddc45ad08e6a2c885ca2255fc52d6df,canceled,2018-09-13 09:56:12,NaT,NaT,NaT,2018-09-27
24998,392ed9afd714e3c74767d0c4d3e3f477,2823ffda607a2316375088e0d00005ec,canceled,2018-09-29 09:13:03,NaT,NaT,NaT,2018-10-15


- 25,000 total orders -> 24,222 delivered orders -> 24,222 usable orders for late delivery analysis

In [95]:
orders["order_status"].value_counts()

order_status
delivered      24222
shipped          309
unavailable      167
canceled         154
invoiced          75
processing        71
approved           1
created            1
Name: count, dtype: int64

- count order_status when order_delivered_customer_date is missing
- order_delivered_customer_date <-> order_status relationship check
- check whether order_estimated_delivery_date has any missing data

In [96]:
orders["order_delivered_customer_date"].isna().sum()

np.int64(775)

In [97]:
orders[orders["order_delivered_customer_date"].isna()]["order_status"].value_counts()

order_status
shipped        309
unavailable    167
canceled       151
invoiced        75
processing      71
approved         1
created          1
Name: count, dtype: int64

In [98]:
orders["order_estimated_delivery_date"].isna().sum()

np.int64(0)

So far, usable population condition

- order_status == "delivered"
- when order_delivered_customer_date is not missing
- when order_estimated_delivery_date is not missing

In [99]:
usable_orders = orders[
    (orders["order_status"]=="delivered")
    & (orders["order_delivered_customer_date"].notna())
    & (orders["order_estimated_delivery_date"].notna())
]

usable_orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28
5,7033745709b7cf1bac7d2533663592de,7f0ca17bb33b230b47459437cf0682c7,delivered,2016-10-04 14:13:22,2016-10-04 14:46:48,2016-10-08 14:46:49,2016-10-11 14:46:49,2016-11-30
...,...,...,...,...,...,...,...,...
24986,9344079d5fbb7dd79af908b7eb667db7,be169d57b327a59f2db49cffdc6caf13,delivered,2018-08-28 15:11:49,2018-08-29 03:15:39,2018-08-29 09:47:00,2018-08-30 17:28:28,2018-09-03
24987,2c1f8fd9aa5f50ec17f4c95e52223a3a,0b0e8ff0c3d490e69901bd6f0efa190a,delivered,2018-08-28 15:53:09,2018-08-28 16:05:19,2018-08-29 19:11:00,2018-08-30 15:11:38,2018-09-06
24988,6d6783e9e03bc56ba0d4f018d45babe3,eb9f43bc3cb1915ba52ea755eddad3d4,delivered,2018-08-28 19:27:43,2018-08-28 19:43:57,2018-08-29 19:02:00,2018-08-30 18:22:47,2018-09-05
24989,d4fae577806d683110e00e18a5e181be,49a6ae8a95c6a78d90945b983ab1ecfc,delivered,2018-08-28 19:32:05,2018-08-29 03:15:36,2018-08-29 13:00:00,2018-08-30 18:32:36,2018-09-12


### B. Olist Tables + Candidate Variables

- **Late-delivery risk:** Which order characteristics may be related to late delivery?
- **Geographic / fulfillment:** Which data may help interpret geographic delivery patterns?

In [100]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28


In [101]:
candidate_tables = {
    "orders": orders,
    "customers": customers,
    "items": items,
    "products": products,
    "sellers": sellers,
    "geolocation": geo
}

for name, df in candidate_tables.items():
    print(f"\n{name}")
    print(df.columns.tolist())


orders
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

customers
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

items
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

products
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

sellers
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

geolocation
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state', 'geolocation_records']


- **Orders**
  - Purchase timing
  - Delivery-related dates
  - Late-delivery outcome

- **Customers**
  - Customer state
  - Customer ZIP-code prefix

- **Order Items**
  - Item count
  - Seller
  - Product
  - Price
  - Freight value

- **Products**
  - Product category
  - Product weight
  - Product dimensions

- **Sellers**
  - Seller state
  - Seller ZIP-code prefix

- **Geolocation**
  - ZIP-code prefix
  - Latitude and longitude

- These variables provide potential information for both late-delivery risk and geographic delivery analysis.

### C. Variable Timing + Potential Leakage

- Candidate variables should be checked based on when the information becomes available.
- Variables used to identify delivery risk should not include information that becomes available only after the delivery outcome is known.
- Check the timing-related fields in the `orders` table.

In [102]:
orders[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].head()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24
3,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28
4,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28


- **Potential pre-delivery information**
  - Purchase timing
  - Customer location
  - Seller location
  - Product characteristics
  - Item price and freight information
  - Estimated delivery date

- **Post-purchase / outcome information**
  - Actual customer delivery date
  - Final order status
  - Review score and review information

- `order_delivered_customer_date` is needed to define the late-delivery outcome, but it should not be used as a predictor of that outcome.
- Review information should also not be used to identify delivery risk because it is generated after the customer receives the order.

### D. Unit of Analysis + Aggregation Plan

- **Final unit of analysis:** One order
- `orders` will be the base table because one row represents one order.
- Supporting tables may contain multiple rows for the same `order_id`.
- Check the number of rows and unique orders before planning future integration.

In [103]:
print("Orders")
print("Rows:", len(orders))
print("Unique order IDs:", orders["order_id"].nunique())

print("\nItems")
print("Rows:", len(items))
print("Unique order IDs:", items["order_id"].nunique())

print("\nPayments")
print("Rows:", len(payments))
print("Unique order IDs:", payments["order_id"].nunique())

print("\nReviews")
print("Rows:", len(reviews))
print("Unique order IDs:", reviews["order_id"].nunique())

Orders
Rows: 25000
Unique order IDs: 25000

Items
Rows: 28343
Unique order IDs: 24791

Payments
Rows: 26154
Unique order IDs: 25000

Reviews
Rows: 24934
Unique order IDs: 24797


- `orders` has one row per order, while the supporting tables can contain multiple records for the same order.
- Therefore, supporting tables will need to be summarized to the order level before integration.

### E. Geographic + Fulfillment Feasibility

- Customer and seller locations may help identify geographic delivery patterns.
- Check the geographic coverage available in the dataset.

In [104]:
print("Customer states:", customers["customer_state"].nunique())
print("Seller states:", sellers["seller_state"].nunique())
print("Geolocation states:", geo["geolocation_state"].nunique())

Customer states: 27
Seller states: 21
Geolocation states: 27


In [105]:
customers["customer_state"].value_counts()

customer_state
SP    10478
RJ     3313
MG     2933
RS     1403
PR     1262
SC      860
BA      851
ES      524
DF      505
GO      498
PE      417
CE      329
PA      221
MT      212
MS      182
MA      160
PB      147
PI      130
RN      120
AL      107
SE      100
TO       74
RO       72
AM       42
AC       25
AP       20
RR       15
Name: count, dtype: int64

In [106]:
sellers["seller_state"].value_counts()

seller_state
SP    1362
PR     247
MG     170
SC     136
RJ     119
RS      84
DF      27
GO      26
ES      16
BA      13
CE       7
PE       6
RN       4
PB       4
MS       4
MT       3
RO       2
PI       1
SE       1
MA       1
PA       1
Name: count, dtype: int64

In [107]:
geo[
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng"
    ]
].head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng
0,1001,-23.550190,-46.634024
1,1004,-23.549799,-46.634757
2,1005,-23.549456,-46.636733
3,1006,-23.550102,-46.636137
4,1008,-23.546002,-46.635886


- Customer and seller location data provide enough geographic information to examine delivery patterns across areas.
- These patterns can help identify areas with delivery challenges, but cannot determine an optimal facility location by themselves.

### F. Expected Relationships + Analytical Plan

- Examine whether order characteristics and geographic patterns are associated with late delivery.
- Use order-level information from the supporting tables after appropriate aggregation.
- Compare late-delivery patterns across customer and seller locations.
- Start with descriptive analysis before moving to later modeling.

## 5. Early Privacy, Ethics, and Limitations

### Privacy and Ethics

- Customer and seller geographic information should be used only for aggregated analysis, not to identify individuals.
- Geographic differences in late delivery should not automatically be interpreted as poor performance by customers, sellers, or specific areas.
- The results should be used to support further operational analysis rather than make decisions automatically.

### Limitations

- The dataset does not include factors such as facility cost, labor availability, warehouse capacity, or detailed transportation information.
- Missing data and differences in table structure may affect the final analytical dataset.
- Relationships found in the analysis should be interpreted as associations, not causation.
- Geographic delivery patterns alone cannot determine the best location for additional fulfillment capacity.

## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what key defines one row, and which tables were joined or aggregated.

Critical integration reminders:

- Enrich item records with product, category, and seller attributes while they are still item-level.
- Aggregate item, payment, and review records separately to `order_id`.
- Join each order-level summary to `orders` with merge validation.
- Treat the order-level ABT as trustworthy only after its row count and unique key have been checked.

The starter below demonstrates this sequence. Revise the variables to fit your question, and explain why each retained variable is useful.

### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



### 1. Data Quality Investigation and Cleaning

In Milestone 1, I identified five data quality issues and developed an initial cleaning plan. In this section, I will investigate these issues in the same order and apply cleaning steps when needed.

I will keep the original data unchanged, use separate working copies, and document each cleaning decision before building the preliminary ABT.

In [112]:
# Create working copies for Milestone 2
# Keep the original Milestone 1 data unchanged

orders_clean = orders.copy()
customers_clean = customers.copy()
items_clean = items.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
geo_clean = geo.copy()
translations_clean = translations.copy()

print("Working copies created.")

Working copies created.


### Data Integration and Preliminary ABT

**Project Direction**

This milestone builds on the data profiling from Milestone 1 to examine factors associated with late delivery. The goal is to integrate order, product, payment, and geographic information into a preliminary Analytical Base Table (ABT).

**ABT Design**

The unit of analysis is one order, identified by `order_id`. The `orders` table serves as the base table, and supporting data will be integrated at the order level to preserve one row per order.

The following sections document the cleaning, aggregation, integration, and validation steps alongside the corresponding code and results.

# 1. Standardize ZIP code prefixes as 5-character strings
zip_columns = [
    (customers, "customer_zip_code_prefix"),
    (sellers, "seller_zip_code_prefix"),
    (geo, "geolocation_zip_code_prefix"),
]

for df, col in zip_columns:
    df[col] = df[col].astype("string").str.strip().str.zfill(5)

#### Issue 1: ZIP Code Data Types

Milestone 1 showed that ZIP code prefixes in the customers, sellers, and geolocation tables were stored as integers.

Since ZIP code prefixes are geographic identifiers rather than numerical measurements, I will convert them to five-character strings. I will then check the data types, missing values, and ZIP code formats to validate the conversion.

In [114]:
# Issue 1: ZIP Code Data Types

zip_columns = [
    (customers_clean, "customer_zip_code_prefix"),
    (sellers_clean, "seller_zip_code_prefix"),
    (geo_clean, "geolocation_zip_code_prefix")
]

# 1. Check data types before cleaning
print("BEFORE CLEANING")

for df, col in zip_columns:
    print(f"{col}: {df[col].dtype}")
    print("Missing values:", df[col].isna().sum())
    print("Sample values:", df[col].head(5).tolist())
    print()

# 2. Convert ZIP code prefixes to five-character strings
for df, col in zip_columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.zfill(5)
    )

# 3. Validate the results
print("AFTER CLEANING")

for df, col in zip_columns:
    invalid_format = (
        df[col].notna()
        & ~df[col].str.fullmatch(r"\d{5}").fillna(False)
    )

    print(f"\n{col}")
    print("Data type:", df[col].dtype)
    print("Missing values:", df[col].isna().sum())
    print("Invalid ZIP formats:", invalid_format.sum())
    print("Sample values:", df[col].head(5).tolist())

BEFORE CLEANING
customer_zip_code_prefix: int64
Missing values: 0
Sample values: [14409, 1151, 13056, 4534, 81560]

seller_zip_code_prefix: int64
Missing values: 0
Sample values: [13023, 13844, 4195, 55325, 16304]

geolocation_zip_code_prefix: int64
Missing values: 0
Sample values: [1001, 1004, 1005, 1006, 1008]

AFTER CLEANING

customer_zip_code_prefix
Data type: string
Missing values: 0
Invalid ZIP formats: 0
Sample values: ['14409', '01151', '13056', '04534', '81560']

seller_zip_code_prefix
Data type: string
Missing values: 0
Invalid ZIP formats: 0
Sample values: ['13023', '13844', '04195', '55325', '16304']

geolocation_zip_code_prefix
Data type: string
Missing values: 0
Invalid ZIP formats: 0
Sample values: ['01001', '01004', '01005', '01006', '01008']


**Findings and Cleaning Decision**

All three ZIP code prefix columns were successfully converted from integers to five-character strings. Leading zeros were restored where needed, and no missing values or invalid ZIP formats were found after conversion.

I will keep the cleaned ZIP code prefixes as strings for geographic matching and data integration.

#### Issue 2: Missing Delivery-Related Dates

Milestone 1 identified missing actual delivery dates in the orders table. I will examine these missing values by order status and check which orders have enough information to define the late delivery outcome.

I will not fill in missing delivery dates because the orders may not have been delivered.

In [115]:
# Issue 2: Missing Delivery-Related Dates

delivery_audit = orders_clean.assign(
    missing_actual_date=orders_clean[
        "order_delivered_customer_date"
    ].isna(),
    missing_estimated_date=orders_clean[
        "order_estimated_delivery_date"
    ].isna()
)

# Check missing delivery dates by order status
delivery_summary = (
    delivery_audit
    .groupby("order_status", dropna=False)
    .agg(
        total_orders=("order_id", "size"),
        missing_actual_dates=("missing_actual_date", "sum"),
        missing_estimated_dates=("missing_estimated_date", "sum")
    )
)

display(delivery_summary)

# Check orders with actual delivery dates but non-delivered status
exceptions = delivery_audit[
    delivery_audit["order_delivered_customer_date"].notna()
    & delivery_audit["order_status"].ne("delivered")
]

print(
    "Actual delivery date present but status is not delivered:",
    len(exceptions)
)

display(
    exceptions[
        ["order_id", "order_status", "order_delivered_customer_date"]
    ]
)

,total_orders,missing_actual_dates,missing_estimated_dates
order_status,,,
approved,1,1,0
canceled,154,151,0
created,1,1,0
delivered,24222,0,0
invoiced,75,75,0
processing,71,71,0
shipped,309,309,0
unavailable,167,167,0


Actual delivery date present but status is not delivered: 3


,order_id,order_status,order_delivered_customer_date
68,dabf2b0e35b423f94618bf965fcb7514,canceled,2016-10-16 14:36:59
72,2c45c33d2f9cb8ff8b1c86cc28c11c30,canceled,2016-11-09 14:53:50
14250,1950d777989f6a877539f53795b4c3c3,canceled,2018-03-21 22:03:51


**Findings and Cleaning Decision**

Out of 25,000 orders, 775 had missing actual delivery dates, while none had missing estimated delivery dates. All 24,222 delivered orders had both dates available.

I also found three canceled orders with actual delivery dates. Since their order status was not "delivered," I will not include them in the late delivery outcome.

I will keep all original records and will not fill in missing delivery dates. The late delivery target will only be defined for the 24,222 delivered orders with both dates available. The remaining 778 orders will have an undefined late delivery target.

#### Issue 3: Item Price and Freight Outliers

Milestone 1 identified unusually high item prices and freight values in the order_items table.

I will examine their distributions and extreme values to determine whether they appear to be data quality issues. I will not remove high values without evidence that they are incorrect.

In [116]:
# Issue 3: Item Price and Freight Outliers

# 1. Summary statistics
display(
    items_clean[["price", "freight_value"]]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.95, 0.99])
)

# 2. Check missing, zero, and negative values
for col in ["price", "freight_value"]:
    print(f"\n{col}")
    print("Missing:", items_clean[col].isna().sum())
    print("Zero:", (items_clean[col] == 0).sum())
    print("Negative:", (items_clean[col] < 0).sum())

# 3. Identify high-end outliers using IQR
for col in ["price", "freight_value"]:
    q1 = items_clean[col].quantile(0.25)
    q3 = items_clean[col].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    outliers = items_clean[items_clean[col] > upper_bound]

    print(f"\n{col} Outlier Check")
    print("Upper bound:", round(upper_bound, 2))
    print("High outlier count:", len(outliers))
    print("Percentage:", round(len(outliers) / len(items_clean) * 100, 2))

# 4. Inspect the most extreme records
print("\nTOP 10 ITEM PRICES")
display(
    items_clean.nlargest(10, "price")[
        ["order_id", "order_item_id", "product_id",
         "seller_id", "price", "freight_value"]
    ]
)

print("\nTOP 10 FREIGHT VALUES")
display(
    items_clean.nlargest(10, "freight_value")[
        ["order_id", "order_item_id", "product_id",
         "seller_id", "price", "freight_value"]
    ]
)

,price,freight_value
count,28343.000000,28343.000000
mean,120.529931,19.973035
std,178.492390,15.387608
min,2.200000,0.000000
25%,39.900000,13.125000
50%,75.000000,16.320000
75%,135.000000,21.235000
95%,349.900000,45.090000
99%,879.770000,83.754800
max,6735.000000,339.590000



price
Missing: 0
Zero: 0
Negative: 0

freight_value
Missing: 0
Zero: 108
Negative: 0

price Outlier Check
Upper bound: 277.65
High outlier count: 2121
Percentage: 7.48

freight_value Outlier Check
Upper bound: 33.4
High outlier count: 2920
Percentage: 10.3

TOP 10 ITEM PRICES


,order_id,order_item_id,product_id,seller_id,price,freight_value
885,0812eb902a67711a1cb742b3cdaa65ae,1,489ae2aa008f021502940f251d4cce7f,e3b4998c7a498169dc7bce44e6bb6277,6735.00,194.31
7289,426a9742b533fc6fed17d1fd6d143d7e,1,a1beef8f3992dbd4cd8726796aa69c53,512d298ac2a96d1931b6bd30aa21f61d,4399.87,113.45
17450,9de73f3e6157169ad6c32b9f313c7dcb,1,0eb41884a8773e019708fb8409fd8347,c510bc1718f0f2961eaa42a23330681a,3899.00,135.44
3216,1d54db601b417ccdfd3b7071955e6cc7,1,6e729bd456c54469a9af4c8774f1eab7,039e6ad9dae79614493083e241147386,3105.00,104.72
19546,b0b0d3285e59abf2f6c9d7e1bf761323,1,470433f95ba906e17efac3fce39e9ffd,e2a1ac9bf33e5549a2a4f834e70df2f8,2999.89,44.23
23607,d3fc45461660252856d603ed31eb4e77,1,797897e182469fff3c330560d2fc851f,d63c73efd41eb002280e7ec831424edb,2899.00,39.17
6040,3745e06e7f01e1652cfd4b9d31651820,1,e51071c4622d7a15494656e912c8d55b,2bf6a2c1e71bbd29a4ad64e6d3c3629f,2700.00,34.66
24680,de0d8b8e774b3211e7e001d3e5985470,1,e63cb7bb4e916e643c1d4785da2b2846,d63c73efd41eb002280e7ec831424edb,2699.00,129.42
1680,0f924038b00d1f102e4d09bea4fa666a,1,86999f823d7791c5796d506ae0ed44ea,eeb6de78f79159600292e314a77cbd18,2689.00,70.95
14500,83e6338b5cf25dcf222551cb8da8d0d6,1,122bedead6e7541d311e6e0ab3cf248f,961dff0a659b4561334372337dd897d9,2649.00,43.82



TOP 10 FREIGHT VALUES


,order_id,order_item_id,product_id,seller_id,price,freight_value
17602,9f49bd16053df810384e793386312674,1,256a9c364b75753b97bee410c9491ad8,5c030029b5916fed0986310385ec9009,1149.00,339.59
4157,264a7e199467906c0727394df82d1a6a,1,97c948ebc8c04b26b7bbb095d4228f2a,17f51e7198701186712e53a39c564617,1050.00,338.30
1243,0b6230647ed16f4b3e70282dc4b5b87f,1,46e24ce614899e36617e37ea1e4aa6ff,17f51e7198701186712e53a39c564617,1050.00,321.88
28138,fe12e676fd493885d417a34bc3917411,1,1d16c0509e7011656e03c6a582ff6667,17f51e7198701186712e53a39c564617,760.00,312.41
5065,2eb868f2f7abdf4284337d1cdeb7399b,1,c1e4bd7e1144f99ac576aedc324a5cb3,082e0bf4cb865a6533b1e8e498cc0255,144.00,269.00
4473,292d7ad29c8a2d97eb84d18ab3d38997,1,a8baceb529f7e2a5c770cc5b4e3da35d,17f51e7198701186712e53a39c564617,760.00,255.92
19667,b19f37ba4b4c83791c9bead49f965926,1,c6eea384e62b799e78267eaeea987eaa,ff69aa92bb6b1bf9b8b7a51c2ed9cf8b,1648.00,254.55
1214,0b055b1ac16d0401f76e361ff96926eb,1,16c4e87b98a9370a9cbc3a4658a3f45b,d23019c84ffae2d5ef2270367b8605fc,1890.00,234.78
12906,750465ac3478d2b4483153fcd61b9c13,1,c60da43ed0fc0f33c57242c2f631e99e,52f976b17ea7f2f087f56dcc419328f6,1459.00,233.59
14007,7ef259be178f9f06c72a57add85b71cd,1,f4f4debbcfcafe6858d1e37a1f6e436e,5058e8c1e82653974541e83690655b4a,139.98,232.69


**Findings and Cleaning Decision**

The order_items table contains 28,343 records with no missing or negative item prices or freight values.

Using the IQR method, I identified 2,121 high-price records (7.48%) and 2,920 high-freight records (10.30%). I also found 108 records with zero freight values.

The highest item price was 6,735.00, and the highest freight value was 339.59. After reviewing the extreme records, I did not find direct evidence that these values were data entry errors.

I will keep the original price and freight values without removing or replacing the outliers. However, their accuracy cannot be fully confirmed from the available data.

#### Issue 4: Payment Value Outliers

Milestone 1 identified unusual payment values, including zero, very small, and very large amounts.

I will examine the payment value distribution and investigate extreme records. I will also compare order-level payments with item prices and freight charges before making a cleaning decision.

In [117]:
# Issue 4: Payment Value Outliers

# 1. Summary statistics
display(
    payments_clean["payment_value"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

# 2. Check missing, zero, negative, and small values
print("Missing:", payments_clean["payment_value"].isna().sum())
print("Zero:", (payments_clean["payment_value"] == 0).sum())
print("Negative:", (payments_clean["payment_value"] < 0).sum())
print(
    "Positive values below 1:",
    payments_clean["payment_value"].between(
        0, 1, inclusive="neither"
    ).sum()
)

# 3. IQR outlier check
q1 = payments_clean["payment_value"].quantile(0.25)
q3 = payments_clean["payment_value"].quantile(0.75)
upper_bound = q3 + 1.5 * (q3 - q1)

high_payments = payments_clean[
    payments_clean["payment_value"] > upper_bound
]

print("\nIQR upper bound:", round(upper_bound, 2))
print("High payment records:", len(high_payments))
print(
    "Percentage:",
    round(len(high_payments) / len(payments_clean) * 100, 2)
)

# 4. Inspect extreme payments
print("\nTOP 10 PAYMENTS")
display(
    payments_clean.nlargest(10, "payment_value")
)

print("\nZERO PAYMENTS")
display(
    payments_clean[
        payments_clean["payment_value"] == 0
    ].head(10)
)

count    26154.000000
mean       154.073541
std        208.396268
min          0.000000
25%         56.860000
50%        100.015000
75%        173.247500
95%        439.146000
99%       1048.432200
max       6929.310000
Name: payment_value, dtype: float64

Missing: 0
Zero: 2
Negative: 0
Positive values below 1: 45

IQR upper bound: 347.83
High payment records: 1974
Percentage: 7.55

TOP 10 PAYMENTS


,order_id,payment_sequential,payment_type,payment_installments,payment_value
10326,0812eb902a67711a1cb742b3cdaa65ae,1,credit_card,8,6929.31
17376,426a9742b533fc6fed17d1fd6d143d7e,1,credit_card,10,4513.32
25182,9de73f3e6157169ad6c32b9f313c7dcb,1,credit_card,10,3899.00
3999,912343626f370ead5ef2e85a9d757e22,1,credit_card,10,3242.84
10692,1d54db601b417ccdfd3b7071955e6cc7,1,credit_card,5,3209.72
9597,7813842ae95e8c497fc0233232ae815a,1,voucher,1,3184.34
19828,03310aa823a66056268a3bab36e827fb,1,voucher,1,3184.34
26085,b0b0d3285e59abf2f6c9d7e1bf761323,1,credit_card,10,3044.12
9461,d3fc45461660252856d603ed31eb4e77,1,boleto,1,2938.17
23738,6fd59e3ae7e24c50131f6bc97c4c7776,1,credit_card,10,2913.47



ZERO PAYMENTS


,order_id,payment_sequential,payment_type,payment_installments,payment_value
10917,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
25385,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [118]:
# Issue 4: Compare Payments with Item Totals

# Aggregate payment amounts by order
payment_totals = (
    payments_clean
    .groupby("order_id", as_index=False)
    .agg(payment_total=("payment_value", "sum"))
)

# Aggregate item price and freight by order
item_totals = (
    items_clean
    .assign(
        item_total=lambda df: df["price"] + df["freight_value"]
    )
    .groupby("order_id", as_index=False)
    .agg(expected_item_total=("item_total", "sum"))
)

# Compare orders that have both records
payment_check = payment_totals.merge(
    item_totals,
    on="order_id",
    how="inner",
    validate="one_to_one"
)

payment_check["difference"] = (
    payment_check["payment_total"]
    - payment_check["expected_item_total"]
).round(2)

print("Orders compared:", len(payment_check))
print(
    "Orders with a difference greater than 0.01:",
    (payment_check["difference"].abs() > 0.01).sum()
)

print("\nLARGEST PAYMENT DIFFERENCES")
display(
    payment_check.reindex(
        payment_check["difference"].abs()
        .sort_values(ascending=False)
        .index
    ).head(10)
)

Orders compared: 24791
Orders with a difference greater than 0.01: 75

LARGEST PAYMENT DIFFERENCES


,order_id,payment_total,expected_item_total,difference
18314,bc2c82b0ef78d2252b6176d1972db7c9,303.02,242.01,61.01
17003,af9ffff2ce6b3defd34fd4c78857a379,466.97,413.17,53.80
16650,abf1130bc676c9dcadf91e24f5e30a30,496.89,459.36,37.53
17835,b76190a2c095fba255ab46987545a660,263.24,227.91,35.33
16147,a69aaa894f707814637e9794aa556f0f,211.87,179.35,32.52
15992,a52e6fa2d7c6ecaf73ff5f7f10438339,233.97,207.00,26.97
19735,cabb660c11fb7f6bbb5f454ea09c0b9f,278.70,254.81,23.89
4955,33a7b8ae888999afb219263e9a36fb77,198.19,175.34,22.85
5348,37af0498017bcfbffca8855bdc9ecc6b,193.63,171.32,22.31
6609,44be450f0262cf8f77ea4fcab05dafa3,191.32,169.28,22.04


**Findings and Cleaning Decision**

The order_payments table contains 26,154 payment records. There were no missing or negative payment values, but I found two zero payments and 45 positive payments below 1.

Using the IQR method, 1,974 payment records (7.55%) were identified as high outliers. The highest payment was 6,929.31, which matched the item price and freight charge for that order.

After comparing order-level payment totals with item and freight totals, I found 75 differences greater than 0.01 among 24,791 comparable orders.

I will keep the original payment values because the unusual amounts are not confirmed data errors. The payment differences will be documented as a remaining data quality concern.

#### Issue 5: Repeated Review IDs

Milestone 1 identified repeated review_id values in the order_reviews table.

I will investigate whether these repeated IDs represent duplicate records or reviews associated with different orders. I will also check whether some orders have multiple review records before deciding how to handle them.

In [119]:
# Issue 5: Repeated Review IDs

# 1. Check duplicate review IDs and rows
print("Total review records:", len(reviews_clean))
print("Repeated review_id occurrences:",
      reviews_clean["review_id"].duplicated().sum())
print("Fully duplicated rows:",
      reviews_clean.duplicated().sum())

# 2. Investigate repeated review IDs
review_id_audit = (
    reviews_clean
    .groupby("review_id")
    .agg(
        record_count=("order_id", "size"),
        unique_orders=("order_id", "nunique")
    )
    .query("record_count > 1")
)

print("\nRepeated unique review IDs:", len(review_id_audit))
print(
    "Repeated IDs linked to multiple orders:",
    (review_id_audit["unique_orders"] > 1).sum()
)

display(review_id_audit.head(10))

# 3. Check multiple reviews per order
reviews_per_order = reviews_clean.groupby("order_id").size()

print(
    "\nOrders with multiple review records:",
    (reviews_per_order > 1).sum()
)

# 4. Inspect actual repeated review records
repeated_ids = review_id_audit.index

display(
    reviews_clean[
        reviews_clean["review_id"].isin(repeated_ids)
    ][
        ["review_id", "order_id", "review_score"]
    ].sort_values(["review_id", "order_id"]).head(20)
)

Total review records: 24934
Repeated review_id occurrences: 56
Fully duplicated rows: 0

Repeated unique review IDs: 55
Repeated IDs linked to multiple orders: 55


,record_count,unique_orders
review_id,,
017808d29fd1f942d97e50184dfb4c13,2,2
0254bd905dc677a6078990aad3331a36,2,2
08528f70f579f0c830189efc523d2182,2,2
09b871b3f4f85d3595cb37054a9105f1,2,2
1775bd097716d67420d128df008252eb,2,2
193d73d33b656807084215db91b23b8c,2,2
1e3fb3520c0e1e9c12b0dc943f7d437b,2,2
20361385529351bec8dbcc659449cf2f,2,2
2172867fd5b1a55f98fe4608e1547b4b,2,2



Orders with multiple review records: 136


,review_id,order_id,review_score
13840,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5
24919,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5
24118,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1
5239,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1
10063,08528f70f579f0c830189efc523d2182,03310aa823a66056268a3bab36e827fb,1
2139,08528f70f579f0c830189efc523d2182,7813842ae95e8c497fc0233232ae815a,1
14913,09b871b3f4f85d3595cb37054a9105f1,2d4f4cfa89af135842ed4770de178752,3
6937,09b871b3f4f85d3595cb37054a9105f1,bcf4f139b50c6689378fac7fe255a186,3
17166,1775bd097716d67420d128df008252eb,04a2ba381317648726f46210a46ece95,5
10376,1775bd097716d67420d128df008252eb,0b3ca8c9e3b3199306adb467fcd0e982,5


**Findings and Cleaning Decision**

The order_reviews table contains 24,934 records. I identified 56 repeated review_id occurrences across 55 unique review IDs, but no fully duplicated rows.

All 55 repeated review IDs were linked to different orders. I also found 136 orders with multiple review records.

I will keep the original review records rather than removing them based only on review_id. Before joining the reviews table to the order-level ABT, I will aggregate the review data by order_id to avoid creating duplicate order rows.

### Data Transformation & Feature Engineering

#### Step 1: Product Missingness and Category Translation

Before building the order-level ABT, I will examine missing product attributes and add English product category names.

I will use the category translation table to enrich the product data and check whether the merge changes the number of product records. Missing values will be kept unless there is clear evidence supporting a correction.

In [120]:
# Step 1: Product Missingness and Category Translation

# Check missing product attributes
product_missing = pd.DataFrame({
    "missing_count": products_clean.isna().sum(),
    "missing_pct": (
        products_clean.isna().mean() * 100
    ).round(2)
})

print("PRODUCT MISSING VALUES")
display(product_missing)

# Check translation key uniqueness
print("\nDuplicate translation keys:",
      translations_clean["product_category_name"].duplicated().sum())

# Add English category names
products_enriched = products_clean.merge(
    translations_clean,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

# Validate the merge
print("\nPRODUCT MERGE VALIDATION")
print("Products before:", len(products_clean))
print("Products after:", len(products_enriched))
print("Duplicate product IDs:",
      products_enriched["product_id"].duplicated().sum())

print("\nCATEGORY COVERAGE")
print("Missing original categories:",
      products_enriched["product_category_name"].isna().sum())
print("Missing English categories:",
      products_enriched["product_category_name_english"].isna().sum())

# Inspect unmatched translations
display(
    products_enriched.loc[
        products_enriched["product_category_name"].notna()
        & products_enriched["product_category_name_english"].isna(),
        ["product_id", "product_category_name"]
    ].head(10)
)

PRODUCT MISSING VALUES


,missing_count,missing_pct
product_id,0,0.00
product_category_name,230,1.72
product_name_lenght,230,1.72
product_description_lenght,230,1.72
product_photos_qty,230,1.72
product_weight_g,1,0.01
product_length_cm,1,0.01
product_height_cm,1,0.01
product_width_cm,1,0.01



Duplicate translation keys: 0

PRODUCT MERGE VALIDATION
Products before: 13344
Products after: 13344
Duplicate product IDs: 0

CATEGORY COVERAGE
Missing original categories: 230
Missing English categories: 234


,product_id,product_category_name
656,0105b5323d24fc655f73052694dbbb3a,pc_gamer
3584,bed164d9d628cf0593003389c535c6e0,portateis_cozinha_e_preparadores_de_alimentos
4480,1220978a08a6b29a202bc015b18250e9,portateis_cozinha_e_preparadores_de_alimentos
5795,ae62bb0f95af63d64eae5f93dddea8d3,portateis_cozinha_e_preparadores_de_alimentos


**Findings and Cleaning Decision**

I examined missing product attributes and checked the coverage of English category translations.

I used a LEFT JOIN to add English category names while preserving the original product records. I also checked the product row count and duplicate product IDs to validate the merge.

I will keep missing product attributes unchanged rather than filling them with estimated values. Products without English category names will also be retained.

The enriched product table will be used to add product information to the item-level data before order-level aggregation.

#### Step 2: Late Delivery Target Creation

The main business question focuses on factors associated with late delivery.

I will create a binary late_delivery target by comparing the actual delivery date with the estimated delivery date. The target will only be defined for orders marked as delivered with both dates available.

Orders that do not meet these conditions will have a missing target value. I will preserve all original orders for later data integration.

In [121]:
# Step 2: Late Delivery Target Creation

# Identify eligible delivered orders
eligible_orders = (
    orders_clean["order_status"].eq("delivered")
    & orders_clean["order_delivered_customer_date"].notna()
    & orders_clean["order_estimated_delivery_date"].notna()
)

# Create nullable binary target
orders_clean["late_delivery"] = pd.Series(
    pd.NA, index=orders_clean.index, dtype="Int64"
)

orders_clean.loc[eligible_orders, "late_delivery"] = (
    orders_clean.loc[
        eligible_orders, "order_delivered_customer_date"
    ]
    >
    orders_clean.loc[
        eligible_orders, "order_estimated_delivery_date"
    ]
).astype(int)

# Validate target
print("Total orders:", len(orders_clean))
print("Eligible orders:", eligible_orders.sum())
print("Undefined targets:", orders_clean["late_delivery"].isna().sum())

print("\nLATE DELIVERY DISTRIBUTION")
display(
    orders_clean["late_delivery"]
    .value_counts(dropna=False)
    .rename_axis("late_delivery")
    .to_frame("order_count")
)

print("\nLATE DELIVERY RATE")
print(
    round(
        orders_clean.loc[eligible_orders, "late_delivery"].mean() * 100,
        2
    ),
    "%"
)

Total orders: 25000
Eligible orders: 24222
Undefined targets: 778

LATE DELIVERY DISTRIBUTION


,order_count
late_delivery,
0,22304
1,1918
<NA>,778



LATE DELIVERY RATE
7.92 %


**Findings and Decision**

Out of 25,000 orders, 24,222 delivered orders had both actual and estimated delivery dates available. The remaining 778 orders were not eligible for the late delivery target.

I created a binary `late_delivery` variable, assigning 1 to orders delivered after the estimated date and 0 to orders delivered on or before the estimated date. Ineligible orders were assigned missing target values.

I kept all 25,000 orders in the dataset without filling in missing delivery dates. The `late_delivery` variable will be used as the outcome for the analysis. Actual delivery dates and delivery durations will not be used as predictors to avoid data leakage.

#### Step 3: Delivery-Related Feature Engineering

I will create delivery-related variables to better understand order fulfillment and delivery performance.

I will calculate the actual delivery duration, estimated delivery window, and delivery delay in days.

I will keep missing values where the required dates are unavailable. Since actual delivery information is only known after fulfillment, these variables will be used for descriptive analysis rather than as predictors of late delivery.

In [122]:
# Step 3: Delivery-Related Feature Engineering

# Ensure date columns use datetime format
date_columns = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col], errors="coerce"
    )

# 1. Actual delivery duration (days)
orders_clean["actual_delivery_days"] = (
    orders_clean["order_delivered_customer_date"]
    - orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# 2. Estimated delivery window (days)
orders_clean["estimated_delivery_days"] = (
    orders_clean["order_estimated_delivery_date"]
    - orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# 3. Delivery delay (days)
orders_clean["delivery_delay_days"] = (
    orders_clean["order_delivered_customer_date"]
    - orders_clean["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

# 4. Inspect new variables
print("DELIVERY FEATURE SUMMARY")
display(
    orders_clean[
        [
            "actual_delivery_days",
            "estimated_delivery_days",
            "delivery_delay_days"
        ]
    ].describe()
)

print("\nMISSING VALUES")
display(
    orders_clean[
        [
            "actual_delivery_days",
            "estimated_delivery_days",
            "delivery_delay_days"
        ]
    ].isna().sum()
)

print("\nNEGATIVE DURATION CHECK")
for col in [
    "actual_delivery_days",
    "estimated_delivery_days"
]:
    print(col, ":", (orders_clean[col] < 0).sum())

DELIVERY FEATURE SUMMARY


,actual_delivery_days,estimated_delivery_days,delivery_delay_days
count,24225.000000,25000.000000,24225.000000
mean,12.471031,23.792331,-11.305518
std,9.296112,8.866064,10.068596
min,0.892488,2.026019,-134.308530
25%,6.755891,18.314120,-16.303356
50%,10.196100,23.232865,-12.049583
75%,15.559734,28.474372,-6.477685
max,208.351759,155.135463,188.975081



MISSING VALUES


actual_delivery_days       775
estimated_delivery_days      0
delivery_delay_days        775
dtype: int64


NEGATIVE DURATION CHECK
actual_delivery_days : 0
estimated_delivery_days : 0


**Findings and Feature Engineering Decision**

I created three delivery-related variables using the order purchase, actual delivery, and estimated delivery timestamps.

The average actual delivery duration was 12.47 days, compared with an average estimated delivery window of 23.79 days. The average delivery delay was -11.31 days, indicating that deliveries were generally earlier than estimated.

There were 775 missing values in both actual_delivery_days and delivery_delay_days because actual delivery dates were unavailable. No negative actual or estimated delivery durations were found.

I will keep the original values, including negative delivery delays that represent early deliveries. I will also retain the extreme delivery durations because they have not been confirmed as data errors.

These variables will be used for descriptive analysis. Actual delivery duration and delivery delay will not be used as predictors of late delivery to avoid data leakage.

### Order-Level Aggregation

The supporting tables have different row meanings, so they need to be summarized before joining them to the `orders` table.

First, I joined product and seller information to the item-level data. Then, I grouped `order_items`, `order_payments`, and `order_reviews` separately by `order_id` to calculate order-level summaries.

This prevents orders with multiple items, payments, or reviews from creating duplicate rows in the preliminary ABT.

#### Step 4: Order Items Aggregation

The order_items table can contain multiple rows for the same order because one order may include multiple items.

I will aggregate the item-level data by order_id to calculate the number of items, total item price, and total freight cost for each order.

I will also check whether the aggregation produces one row per order before joining it to the main orders table.

In [123]:
# Step 4: Order Items Aggregation

# 1. Check original item-level data
print("Original item rows:", len(items_clean))
print("Unique orders:", items_clean["order_id"].nunique())

# 2. Aggregate items by order
items_agg = (
    items_clean
    .groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "size"),
        total_item_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique")
    )
)

# 3. Calculate total item and freight cost
items_agg["total_order_value"] = (
    items_agg["total_item_price"]
    + items_agg["total_freight"]
)

# 4. Validate aggregation
print("\nAGGREGATION VALIDATION")
print("Aggregated rows:", len(items_agg))
print("Duplicate order IDs:",
      items_agg["order_id"].duplicated().sum())

print("\nAGGREGATED ITEM SUMMARY")
display(
    items_agg[
        [
            "item_count",
            "total_item_price",
            "total_freight",
            "total_order_value",
            "unique_products",
            "unique_sellers"
        ]
    ].describe()
)

print("\nSAMPLE AGGREGATED ORDERS")
display(items_agg.head(10))

Original item rows: 28343
Unique orders: 24791

AGGREGATION VALIDATION
Aggregated rows: 24791
Duplicate order IDs: 0

AGGREGATED ITEM SUMMARY


,item_count,total_item_price,total_freight,total_order_value,unique_products,unique_sellers
count,24791.000000,24791.000000,24791.000000,24791.000000,24791.000000,24791.000000
mean,1.143278,137.799194,22.834728,160.633922,1.039974,1.014642
std,0.533246,200.688777,20.553735,210.501119,0.233481,0.125055
min,1.000000,2.200000,0.000000,9.590000,1.000000,1.000000
25%,1.000000,45.950000,13.920000,62.230000,1.000000,1.000000
50%,1.000000,88.340000,17.200000,106.160000,1.000000,1.000000
75%,1.000000,149.990000,24.030000,178.490000,1.000000,1.000000
max,12.000000,6735.000000,436.240000,6929.310000,8.000000,3.000000



SAMPLE AGGREGATED ORDERS


,order_id,item_count,total_item_price,total_freight,unique_products,unique_sellers,total_order_value
0,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,1,1,216.87
1,00048cc3ae777c65dbb7d2a0634bc1ea,1,21.90,12.69,1,1,34.59
2,00054e8431b9d7675808bcb819fb4a32,1,19.90,11.85,1,1,31.75
3,0005a1a1728c9d785b8e2b08b904576c,1,145.95,11.65,1,1,157.60
4,0009792311464db532ff765bf7b182ae,1,99.90,27.65,1,1,127.55
5,000aed2e25dbad2f9ddb70584c5a2ded,1,144.00,8.77,1,1,152.77
6,000f25f4d72195062c040b12dce9a18a,1,119.99,44.40,1,1,164.39
7,001021efaa8636c29475e7734483457d,1,49.00,15.10,1,1,64.10
8,00130c0eee84a3d909e75bc08c5c3ca1,1,27.90,7.94,1,1,35.84
9,001427c0ec99cf8af737bd88e92fd444,1,59.90,23.35,1,1,83.25


**Findings and Aggregation Decision**

I aggregated the order_items table by order_id to create one record per order.

For each order, I calculated the number of items, total item price, total freight cost, number of unique products, and number of unique sellers. I also created total_order_value by adding item prices and freight charges.

I kept the original item-level records unchanged and created a separate aggregated table named items_agg.

This table will be used in the final ABT to analyze how order size, value, and seller characteristics are associated with late delivery.

#### Step 5: Order Payments Aggregation

The order_payments table may contain multiple payment records for the same order.

I will aggregate payment information by order_id to calculate the total payment amount, number of payment records, and maximum number of installments.

I will validate that the aggregated table contains only one row per order before joining it to the final ABT.

In [124]:
# Step 5: Order Payments Aggregation

# 1. Check original payment data
print("Original payment rows:", len(payments_clean))
print("Unique orders:", payments_clean["order_id"].nunique())

# 2. Aggregate payments by order
payments_agg = (
    payments_clean
    .groupby("order_id", as_index=False)
    .agg(
        total_payment=("payment_value", "sum"),
        payment_count=("payment_value", "size"),
        max_installments=("payment_installments", "max"),
        payment_type_count=("payment_type", "nunique")
    )
)

# 3. Validate aggregation
print("\nAGGREGATION VALIDATION")
print("Aggregated rows:", len(payments_agg))
print("Duplicate order IDs:",
      payments_agg["order_id"].duplicated().sum())

print("\nAGGREGATED PAYMENT SUMMARY")
display(
    payments_agg[
        [
            "total_payment",
            "payment_count",
            "max_installments",
            "payment_type_count"
        ]
    ].describe()
)

print("\nSAMPLE AGGREGATED ORDERS")
display(payments_agg.head(10))

Original payment rows: 26154
Unique orders: 25000

AGGREGATION VALIDATION
Aggregated rows: 25000
Duplicate order IDs: 0

AGGREGATED PAYMENT SUMMARY


,total_payment,payment_count,max_installments,payment_type_count
count,25000.000000,25000.000000,25000.000000,25000.000000
mean,161.185575,1.046160,2.955840,1.022640
std,212.532565,0.394202,2.732843,0.148756
min,9.590000,1.000000,1.000000,1.000000
25%,62.405000,1.000000,1.000000,1.000000
50%,106.215000,1.000000,2.000000,1.000000
75%,178.892500,1.000000,4.000000,1.000000
max,6929.310000,22.000000,24.000000,2.000000



SAMPLE AGGREGATED ORDERS


,order_id,total_payment,payment_count,max_installments,payment_type_count
0,000229ec398224ef6ca0657da4fc703e,216.87,1,5,1
1,00048cc3ae777c65dbb7d2a0634bc1ea,34.59,1,1,1
2,00054e8431b9d7675808bcb819fb4a32,31.75,1,1,1
3,0005a1a1728c9d785b8e2b08b904576c,157.60,1,3,1
4,0009792311464db532ff765bf7b182ae,127.55,1,1,1
5,000aed2e25dbad2f9ddb70584c5a2ded,152.77,1,1,1
6,000f25f4d72195062c040b12dce9a18a,164.39,1,1,1
7,001021efaa8636c29475e7734483457d,64.10,1,3,1
8,00130c0eee84a3d909e75bc08c5c3ca1,35.84,1,1,1
9,001427c0ec99cf8af737bd88e92fd444,83.25,1,1,1


**Findings and Aggregation Decision**

I aggregated the order_payments table by order_id to create one payment summary per order.

I calculated the total payment amount, number of payment records, maximum installments, and number of unique payment types for each order.

I kept the original payment records unchanged and created a separate aggregated table named payments_agg.

This aggregation will allow payment information to be joined to the order-level ABT without creating duplicate order rows.

#### Step 6: Order Reviews Aggregation

The order_reviews table may contain multiple review records for the same order. Milestone 1 also identified repeated review_id values.

I will aggregate the review data by order_id to calculate the average review score and number of review records for each order.

I will keep the original review records and check that the aggregated table contains only one row per order.

In [125]:
# Step 6: Order Reviews Aggregation

# 1. Check original review data
print("Original review rows:", len(reviews_clean))
print("Unique orders:", reviews_clean["order_id"].nunique())

# 2. Aggregate reviews by order
reviews_agg = (
    reviews_clean
    .groupby("order_id", as_index=False)
    .agg(
        avg_review_score=("review_score", "mean"),
        review_count=("review_id", "size"),
        min_review_score=("review_score", "min"),
        max_review_score=("review_score", "max")
    )
)

# 3. Validate aggregation
print("\nAGGREGATION VALIDATION")
print("Aggregated rows:", len(reviews_agg))
print(
    "Duplicate order IDs:",
    reviews_agg["order_id"].duplicated().sum()
)

print(
    "Original review rows preserved in counts:",
    reviews_agg["review_count"].sum() == len(reviews_clean)
)

# 4. Review summary
print("\nAGGREGATED REVIEW SUMMARY")
display(
    reviews_agg[
        [
            "avg_review_score",
            "review_count",
            "min_review_score",
            "max_review_score"
        ]
    ].describe()
)

# 5. Inspect orders with multiple reviews
print("\nORDERS WITH MULTIPLE REVIEWS")
display(
    reviews_agg[
        reviews_agg["review_count"] > 1
    ].head(10)
)

Original review rows: 24934
Unique orders: 24797

AGGREGATION VALIDATION
Aggregated rows: 24797
Duplicate order IDs: 0
Original review rows preserved in counts: True

AGGREGATED REVIEW SUMMARY


,avg_review_score,review_count,min_review_score,max_review_score
count,24797.000000,24797.000000,24797.000000,24797.000000
mean,4.096967,1.005525,4.095011,4.098923
std,1.341337,0.074667,1.343614,1.340972
min,1.000000,1.000000,1.000000,1.000000
25%,4.000000,1.000000,4.000000,4.000000
50%,5.000000,1.000000,5.000000,5.000000
75%,5.000000,1.000000,5.000000,5.000000
max,5.000000,3.000000,5.000000,5.000000



ORDERS WITH MULTIPLE REVIEWS


,order_id,avg_review_score,review_count,min_review_score,max_review_score
138,0176a6846bcb3b0d3aa3116a9a768597,5.0,2,5,5
308,03515a836bb855b03f7df9dee520a8fc,5.0,2,5,5
683,0715dfcf2383aa72c181d8b47f6cb589,5.0,2,5,5
715,076d991923ee938761994a4255f5c5da,5.0,2,5,5
1057,0b127810f86631fa896fd74a421045f8,5.0,2,5,5
1071,0b4aab4ef70208498663141571f68116,1.0,2,1,1
1346,0e457aee274ec7e2e18d25ab6d212921,5.0,2,5,5
1517,10467788e05701fc7e9293f6cbed5c11,5.0,2,5,5
1566,10f288cd8e6f4add2454d845e9df5af6,5.0,2,5,5
1888,142b0e3de54c110a3109b26c79d6ec2e,5.0,2,5,5


**Findings and Aggregation Decision**

I aggregated the order_reviews table by order_id to create one review summary per order.

I calculated the average, minimum, and maximum review scores, along with the number of review records for each order.

I kept the original review records unchanged and created a separate aggregated table named reviews_agg.

This approach handles multiple reviews per order without removing records based on repeated review_id values. The aggregated table will be joined to the order-level ABT without creating duplicate order rows.

### Data Integration: Building the Order-Level ABT

I used LEFT JOINs to combine customer location information and the aggregated item, payment, and review summaries with the `orders` table.

The customer location data was joined using ZIP code prefixes, and customer records were connected to orders using `customer_id`. The item, payment, and review summaries were joined using `order_id`.

I used merge validation to check the expected key relationships and preserve one row per order. LEFT JOINs also allowed me to keep orders even when some supporting information was missing.

In [128]:
# Build the Order-Level ABT

# 1. Prepare geographic information
# Each ZIP prefix should have one geographic summary

if geo_clean["geolocation_zip_code_prefix"].duplicated().any():
    geo_summary = (
        geo_clean
        .groupby("geolocation_zip_code_prefix", as_index=False)
        .agg(
            geolocation_lat=("geolocation_lat", "mean"),
            geolocation_lng=("geolocation_lng", "mean"),
            geolocation_records=("geolocation_lat", "size")
        )
    )
else:
    geo_summary = geo_clean.copy()

    if "geolocation_records" not in geo_summary.columns:
        geo_summary["geolocation_records"] = 1

# 2. Join customer and geographic information
customer_geo = customers_clean.merge(
    geo_summary[[
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_records"
    ]],
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    validate="many_to_one"
).rename(columns={
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng",
    "geolocation_records": "customer_geo_source_records"
})

# 3. Start with cleaned orders
abt = orders_clean.copy()

print("Original orders:", len(abt))

# 4. Join customer information
abt = abt.merge(
    customer_geo,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print("After customer JOIN:", len(abt))

# 5. Join aggregated items
abt = abt.merge(
    items_agg,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("After items JOIN:", len(abt))

# 6. Join aggregated payments
abt = abt.merge(
    payments_agg,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("After payments JOIN:", len(abt))

# 7. Join aggregated reviews
abt = abt.merge(
    reviews_agg,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("After reviews JOIN:", len(abt))

# 8. Create purchase date features
abt["purchase_month"] = (
    abt["order_purchase_timestamp"].dt.month
)

abt["purchase_dayofweek"] = (
    abt["order_purchase_timestamp"].dt.dayofweek
)

# 9. Create review outcome
abt["high_review_score"] = pd.Series(
    pd.NA,
    index=abt.index,
    dtype="Int64"
)

review_observed = abt["avg_review_score"].notna()

abt.loc[review_observed, "high_review_score"] = (
    abt.loc[review_observed, "avg_review_score"] >= 4
).astype(int)

# 10. Preview the ABT
print("\nFINAL ABT")
print("Rows:", len(abt))
print("Unique orders:", abt["order_id"].nunique())
print(
    "Duplicate order IDs:",
    abt["order_id"].duplicated().sum()
)

display(abt.head())

Original orders: 25000
After customer JOIN: 25000
After items JOIN: 25000
After payments JOIN: 25000
After reviews JOIN: 25000

FINAL ABT
Rows: 25000
Unique orders: 25000
Duplicate order IDs: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,late_delivery,actual_delivery_days,estimated_delivery_days,delivery_delay_days,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,customer_lat,customer_lng,customer_geo_source_records,item_count,total_item_price,total_freight,unique_products,unique_sellers,total_order_value,total_payment,payment_count,max_installments,payment_type_count,avg_review_score,review_count,min_review_score,max_review_score,purchase_month,purchase_dayofweek,high_review_score
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27,0,23.178738,23.593866,-0.415127,32ea3bdedab835c3aa6cb68ce66565ef,04106,sao paulo,SP,04106,-23.581451,-46.635029,45.0,1.0,29.90,15.56,1.0,1.0,45.46,45.46,1,1,1,4.0,1.0,4.0,4.0,10,0,1
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23,0,10.733669,50.061447,-39.327778,87776adb449c551e74c13fc34f036105,12030,taubate,SP,12030,-23.032142,-45.570461,218.0,1.0,29.99,10.96,1.0,1.0,40.95,40.95,1,4,1,5.0,1.0,5.0,5.0,10,0,1
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24,<NA>,NaN,50.579340,NaN,823c47d4abda1f8ce7568145f76c2b85,13175,sumare,SP,13175,-22.817992,-47.241101,173.0,NaN,NaN,NaN,NaN,NaN,NaN,50.74,1,3,1,2.0,1.0,2.0,2.0,10,1,0
3,3f72d2b757e725cd48a4726f831c7789,e0de521fbb397bd7e83d079fe66357b5,delivered,2016-10-04 13:15:46,2016-10-04 13:47:04,2016-10-10 16:45:50,2016-10-17 11:25:59,2016-11-28,0,12.923762,54.447384,-41.523623,0829f7df6577d5a4b65439bea701405f,31210,belo horizonte,MG,31210,-19.906712,-43.948897,156.0,1.0,249.90,17.59,1.0,1.0,267.49,267.49,1,10,1,3.0,1.0,3.0,3.0,10,1,0
4,63638a6806d67773f3adba8534553fff,16e14c1e6e050fe6730c961ff638ca23,delivered,2016-10-04 13:22:56,2016-10-04 13:47:45,2016-11-17 15:53:01,2016-11-25 13:17:37,2016-11-28,0,51.996308,54.442407,-2.446100,df2988ba3ed226b10521a0e4da849b61,23092,rio de janeiro,RJ,23092,-22.870856,-43.543849,217.0,1.0,67.90,18.98,1.0,1.0,86.88,86.88,7,1,1,5.0,1.0,5.0,5.0,10,1,1


In [129]:
# Validate the Order-Level ABT

# 1. Row count and duplicate key validation
abt_validation = pd.Series({
    "orders_before_merges": len(orders_clean),
    "abt_rows_after_merges": len(abt),
    "unique_order_ids": abt["order_id"].nunique(),
    "duplicate_order_ids": abt["order_id"].duplicated().sum(),
    "orders_with_item_summary": abt["item_count"].notna().sum(),
    "orders_with_payment_summary": abt["total_payment"].notna().sum(),
    "orders_with_review_summary": abt["avg_review_score"].notna().sum(),
    "orders_with_observed_delivery_target": abt["late_delivery"].notna().sum(),
    "orders_with_observed_review_target": abt["high_review_score"].notna().sum()
})

print("ABT VALIDATION")
display(abt_validation.to_frame("value"))

# 2. Check whether all orders are preserved
print("\nROW COUNT CHECK")
print("Original orders:", len(orders_clean))
print("Final ABT rows:", len(abt))
print("All orders preserved:", len(abt) == len(orders_clean))
print("Duplicate order IDs:", abt["order_id"].duplicated().sum())

# 3. Check missing values
print("\nTOP 15 MISSING VALUE RATES")
display(
    abt.isna().mean()
    .sort_values(ascending=False)
    .head(15)
    .to_frame("missing_rate")
)

# 4. Compare payment totals with item and freight totals
comparable = abt[
    abt["total_payment"].notna()
    & abt["total_order_value"].notna()
].copy()

comparable["payment_difference"] = (
    comparable["total_payment"]
    - comparable["total_order_value"]
)

print("\nPAYMENT VALIDATION")
print("Comparable orders:", len(comparable))
print(
    "Payment mismatches (> $0.01):",
    (comparable["payment_difference"].abs() > 0.01).sum()
)

# 5. Check customer geographic information
print("\nGEOGRAPHIC DATA CHECK")
print("Orders with customer state:",
      abt["customer_state"].notna().sum())
print("Orders with customer latitude:",
      abt["customer_lat"].notna().sum())

# 6. Final shape
print("\nFINAL ABT SHAPE")
print(abt.shape)

ABT VALIDATION


,value
orders_before_merges,25000
abt_rows_after_merges,25000
unique_order_ids,25000
duplicate_order_ids,0
orders_with_item_summary,24791
orders_with_payment_summary,25000
orders_with_review_summary,24797
orders_with_observed_delivery_target,24222
orders_with_observed_review_target,24797



ROW COUNT CHECK
Original orders: 25000
Final ABT rows: 25000
All orders preserved: True
Duplicate order IDs: 0

TOP 15 MISSING VALUE RATES


,missing_rate
late_delivery,0.03112
order_delivered_customer_date,0.03100
delivery_delay_days,0.03100
actual_delivery_days,0.03100
order_delivered_carrier_date,0.01808
total_order_value,0.00836
total_item_price,0.00836
total_freight,0.00836
unique_products,0.00836
unique_sellers,0.00836



PAYMENT VALIDATION
Comparable orders: 24791
Payment mismatches (> $0.01): 95

GEOGRAPHIC DATA CHECK
Orders with customer state: 25000
Orders with customer latitude: 24914

FINAL ABT SHAPE
(25000, 37)


### Row-Count and Duplicate-Key Validation

The final ABT contains 25,000 rows and 37 columns, with no duplicate order IDs. All original orders were preserved after the LEFT JOINs.

Some values were missing, including the late-delivery target (3.11%), item information (0.84%), and review scores (0.81%). Customer location coordinates were available for 99.66% of orders.

I also found payment differences greater than $0.01 in 95 of 24,791 comparable orders. I kept these records and missing values for further investigation rather than removing them.

### SQL Validation Connected to the ABT

I will use PostgreSQL to validate the order-level ABT. I will check the total number of rows, unique order IDs, and available late-delivery targets.

I will compare the SQL results with the Pandas validation results to confirm that the data is consistent.

In [132]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Load Neon connection information
load_dotenv()

database_url = os.getenv("DATABASE_URL")

if not database_url:
    raise ValueError("DATABASE_URL is not set.")

# Connect to Neon PostgreSQL
engine = create_engine(database_url)

with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Neon connection successful:", result.scalar())

Neon connection successful: 1


In [133]:
# Upload the order-level ABT to Neon PostgreSQL
abt.to_sql(
    "olist_order_abt",
    engine,
    if_exists="fail",
    index=False,
    chunksize=1000
)

print("ABT uploaded successfully!")

ABT uploaded successfully!


In [134]:
# Validate the ABT in Neon PostgreSQL
query = """
SELECT
    COUNT(*) AS abt_rows,
    COUNT(DISTINCT order_id) AS unique_order_ids,
    COUNT(late_delivery) AS delivery_target_rows,
    COUNT(high_review_score) AS review_target_rows
FROM olist_order_abt;
"""

sql_abt_check = pd.read_sql_query(query, engine)

print("NEON SQL VALIDATION")
display(sql_abt_check)

NEON SQL VALIDATION


,abt_rows,unique_order_ids,delivery_target_rows,review_target_rows
0,25000,25000,24222,24797


**SQL Validation Findings**

I uploaded the order-level ABT to Neon PostgreSQL and validated it using SQL. The database contains 25,000 orders with no duplicate order IDs. The late-delivery target is available for 24,222 orders, and the review-score target is available for 24,797 orders.

The SQL results match the Pandas validation results, confirming that the ABT was uploaded successfully.

### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [139]:

# Data Dictionary - Order-Level ABT

data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

# Column documentation:
# (description, source, transformation, notes)

documentation = {
    "order_id": (
        "Unique identifier for each order",
        "orders",
        "Kept from the original orders table",
        "Primary key of the order-level ABT"
    ),
    "customer_id": (
        "Customer record identifier associated with the order",
        "orders",
        "Kept from orders and used to join customers",
        "Different from customer_unique_id"
    ),
    "order_status": (
        "Recorded status of the order",
        "orders",
        "Standardized to lowercase and removed extra spaces",
        "May include delivered, canceled, or other statuses"
    ),
    "order_purchase_timestamp": (
        "Date and time when the order was placed",
        "orders",
        "Converted to datetime",
        "Used to calculate purchase timing"
    ),
    "order_approved_at": (
        "Date and time when the order was approved",
        "orders",
        "Converted to datetime",
        "May be missing for some orders"
    ),
    "order_delivered_carrier_date": (
        "Date and time when the order was sent to the carrier",
        "orders",
        "Converted to datetime",
        "May be missing for some orders"
    ),
    "order_delivered_customer_date": (
        "Actual customer delivery date and time",
        "orders",
        "Converted to datetime",
        "Unavailable for orders without recorded delivery"
    ),
    "order_estimated_delivery_date": (
        "Estimated customer delivery deadline",
        "orders",
        "Converted to datetime",
        "Used to evaluate delivery performance"
    ),
    "late_delivery": (
        "Indicator of whether the order was delivered late",
        "orders",
        "Derived from actual and estimated delivery dates",
        "Main delivery outcome; confirm missing-value handling"
    ),
    "actual_delivery_days": (
        "Number of days from purchase to actual delivery",
        "orders",
        "Calculated using purchase and actual delivery dates",
        "Unavailable when actual delivery is missing"
    ),
    "estimated_delivery_days": (
        "Number of days from purchase to estimated delivery",
        "orders",
        "Calculated using purchase and estimated delivery dates",
        "Represents the estimated delivery window"
    ),
    "delivery_delay_days": (
        "Difference between actual and estimated delivery timing",
        "orders",
        "Calculated from actual and estimated delivery dates",
        "Positive values generally indicate late delivery"
    ),
    "customer_unique_id": (
        "Identifier used to recognize the same customer across orders",
        "customers",
        "Joined using customer_id",
        "Useful for identifying repeat customers"
    ),
    "customer_zip_code_prefix": (
        "Five-digit customer ZIP code prefix",
        "customers",
        "Standardized as a five-character string",
        "Used to join geographic information"
    ),
    "customer_city": (
        "City associated with the customer",
        "customers",
        "Joined using customer_id",
        "Useful for geographic comparisons"
    ),
    "customer_state": (
        "State associated with the customer",
        "customers",
        "Standardized to uppercase",
        "Useful for regional delivery analysis"
    ),
    "geolocation_zip_code_prefix": (
        "ZIP code prefix matched to geographic information",
        "geolocation",
        "Joined using customer_zip_code_prefix",
        "May be missing for unmatched ZIP codes"
    ),
    "customer_lat": (
        "Representative latitude for the customer's ZIP prefix",
        "geolocation",
        "Renamed from geolocation_lat",
        "Approximate ZIP-level location"
    ),
    "customer_lng": (
        "Representative longitude for the customer's ZIP prefix",
        "geolocation",
        "Renamed from geolocation_lng",
        "Approximate ZIP-level location"
    ),
    "customer_geo_source_records": (
        "Number of geographic source records for the ZIP prefix",
        "geolocation",
        "Renamed from geolocation_records",
        "Describes geographic data coverage"
    ),
    "item_count": (
        "Number of item records in the order",
        "order_items",
        "Counted item records by order_id",
        "May indicate order size"
    ),
    "total_item_price": (
        "Total price of items in the order",
        "order_items",
        "Summed item prices by order_id",
        "Excludes freight charges"
    ),
    "total_freight": (
        "Total freight charges for the order",
        "order_items",
        "Summed freight values by order_id",
        "Represents shipping charges"
    ),
    "unique_products": (
        "Number of distinct products in the order",
        "order_items",
        "Counted distinct product_id values by order_id",
        "Measures product variety"
    ),
    "unique_sellers": (
        "Number of distinct sellers in the order",
        "order_items",
        "Counted distinct seller_id values by order_id",
        "May reflect fulfillment complexity"
    ),
    "total_order_value": (
        "Total item price including freight charges",
        "order_items",
        "Calculated from total_item_price and total_freight",
        "Compare with total_payment for consistency"
    ),
    "total_payment": (
        "Total recorded payment amount for the order",
        "order_payments",
        "Summed payment_value by order_id",
        "May differ from total_order_value"
    ),
    "payment_count": (
        "Number of payment records associated with the order",
        "order_payments",
        "Counted payment records by order_id",
        "Multiple payments may exist for one order"
    ),
    "max_installments": (
        "Maximum installment count for the order",
        "order_payments",
        "Calculated maximum payment_installments by order_id",
        "Not the sum of installments"
    ),
    "payment_type_count": (
        "Number of distinct payment methods used",
        "order_payments",
        "Counted distinct payment_type values by order_id",
        "An order may have multiple payment types"
    ),
    "avg_review_score": (
        "Average customer review score for the order",
        "order_reviews",
        "Calculated mean review_score by order_id",
        "Post-purchase information; not a purchase-time predictor"
    ),
    "review_count": (
        "Number of review records for the order",
        "order_reviews",
        "Counted review records by order_id",
        "May be missing when no review is matched"
    ),
    "min_review_score": (
        "Lowest review score recorded for the order",
        "order_reviews",
        "Calculated minimum review_score by order_id",
        "Useful when multiple reviews exist"
    ),
    "max_review_score": (
        "Highest review score recorded for the order",
        "order_reviews",
        "Calculated maximum review_score by order_id",
        "Useful when multiple reviews exist"
    ),
    "purchase_month": (
        "Month when the order was purchased",
        "orders",
        "Extracted month from order_purchase_timestamp",
        "Values range from 1 to 12"
    ),
    "purchase_dayofweek": (
        "Day of the week when the order was purchased",
        "orders",
        "Extracted weekday from order_purchase_timestamp",
        "Monday = 0 and Sunday = 6"
    ),
    "high_review_score": (
        "Indicator of whether the average review score is at least 4",
        "order_reviews",
        "Derived from avg_review_score using a threshold of 4",
        "Secondary review outcome; unavailable without a review"
    ),
}

# Fill the dictionary using .loc[]
fields = [
    "description",
    "source",
    "transformation_or_derivation",
    "notes_or_concerns"
]

for column, details in documentation.items():
    data_dictionary.loc[
        data_dictionary["column_name"] == column,
        fields
    ] = list(details)

# Validation
missing_columns = set(abt.columns) - set(documentation)
extra_columns = set(documentation) - set(abt.columns)

incomplete = data_dictionary[
    data_dictionary[fields].isna().any(axis=1)
    | data_dictionary[fields].eq("").any(axis=1)
]

print("ABT columns:", len(abt.columns))
print("Dictionary rows:", len(data_dictionary))
print("Undocumented columns:", sorted(missing_columns))
print("Extra dictionary definitions:", sorted(extra_columns))
print("Incomplete rows:", len(incomplete))

assert not missing_columns
assert not extra_columns
assert incomplete.empty
assert data_dictionary["column_name"].is_unique

display(data_dictionary)


ABT columns: 37
Dictionary rows: 37
Undocumented columns: []
Extra dictionary definitions: []
Incomplete rows: 0


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,order_id,str,Unique identifier for each order,orders,Kept from the original orders table,Primary key of the order-level ABT
1,customer_id,str,Customer record identifier associated with the...,orders,Kept from orders and used to join customers,Different from customer_unique_id
2,order_status,str,Recorded status of the order,orders,Standardized to lowercase and removed extra sp...,"May include delivered, canceled, or other stat..."
3,order_purchase_timestamp,datetime64[us],Date and time when the order was placed,orders,Converted to datetime,Used to calculate purchase timing
4,order_approved_at,datetime64[us],Date and time when the order was approved,orders,Converted to datetime,May be missing for some orders
5,order_delivered_carrier_date,datetime64[us],Date and time when the order was sent to the c...,orders,Converted to datetime,May be missing for some orders
6,order_delivered_customer_date,datetime64[us],Actual customer delivery date and time,orders,Converted to datetime,Unavailable for orders without recorded delivery
7,order_estimated_delivery_date,datetime64[us],Estimated customer delivery deadline,orders,Converted to datetime,Used to evaluate delivery performance
8,late_delivery,Int64,Indicator of whether the order was delivered late,orders,Derived from actual and estimated delivery dates,Main delivery outcome; confirm missing-value h...
9,actual_delivery_days,float64,Number of days from purchase to actual delivery,orders,Calculated using purchase and actual delivery ...,Unavailable when actual delivery is missing


In [138]:
print("Actual ABT columns:")
print(abt.columns.tolist())

Actual ABT columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'late_delivery', 'actual_delivery_days', 'estimated_delivery_days', 'delivery_delay_days', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'geolocation_zip_code_prefix', 'customer_lat', 'customer_lng', 'customer_geo_source_records', 'item_count', 'total_item_price', 'total_freight', 'unique_products', 'unique_sellers', 'total_order_value', 'total_payment', 'payment_count', 'max_installments', 'payment_type_count', 'avg_review_score', 'review_count', 'min_review_score', 'max_review_score', 'purchase_month', 'purchase_dayofweek', 'high_review_score']


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminders:

- If you model `delivered_late`, use only rows where the nullable target is defined; undelivered orders do not reveal a normal late/on-time outcome.
- For a prediction made at purchase time, do not use actual delivery duration, final order status, review fields, or other post-purchase outcomes as predictors.
- If you model `high_review_score`, use only rows with an observed review and do not use review-derived fields as predictors.
- Check class balance, keep a reproducible train/test split, and interpret precision and recall when the positive class is uncommon.
- Explain the intended decision and prediction timing before choosing features.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [108]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [109]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "olist_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "olist_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "olist_model_metrics.csv"
# report_path = REPORTS_DIR / "olist_final_report.html"


In [110]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "olist_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "olist_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
